# Cell 1 — Load the Known-AMR Feature Engineering Cohort

In [1]:
from google.colab import drive
import os
import pandas as pd


# =========================================================
# 1. Mount Google Drive
# =========================================================

drive.mount(
    "/content/drive"
)


# =========================================================
# 2. Project paths
# =========================================================

DRIVE_ROOT = (
    "/content/drive/MyDrive/AMR-Genome-ML"
)

FEATURE_MANIFEST_PATH = os.path.join(
    DRIVE_ROOT,
    "data",
    "processed",
    "genomic_feature_input_manifest.csv"
)

AMRFINDER_PILOT_DIR = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "amrfinder_pilot"
)

AMRFINDER_PILOT_HITS_PATH = os.path.join(
    AMRFINDER_PILOT_DIR,
    "amrfinder_pilot_hits_combined.csv"
)

AMRFINDER_PILOT_METADATA_PATH = os.path.join(
    AMRFINDER_PILOT_DIR,
    "amrfinder_pilot_metadata.csv"
)

AMRFINDER_STRATEGY_PATH = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "amrfinder_screening_strategy_decision.csv"
)


# File 04 output directory
KNOWN_AMR_DIR = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "known_amr"
)

os.makedirs(
    KNOWN_AMR_DIR,
    exist_ok=True
)


# =========================================================
# 3. Verify required persistent files
# =========================================================

required_files = [
    FEATURE_MANIFEST_PATH,
    AMRFINDER_PILOT_HITS_PATH,
    AMRFINDER_PILOT_METADATA_PATH,
    AMRFINDER_STRATEGY_PATH
]


missing_files = [
    path
    for path in required_files
    if not os.path.exists(path)
]


if missing_files:

    raise FileNotFoundError(
        "Missing required File 03 outputs:\n"
        + "\n".join(missing_files)
    )


# =========================================================
# 4. Load finalized cohort
# =========================================================

df_feature_input = pd.read_csv(
    FEATURE_MANIFEST_PATH,
    dtype={
        "Genome ID": str,
        "Taxon ID": str
    }
)


df_feature_input["Genome ID"] = (
    df_feature_input["Genome ID"]
    .astype(str)
    .str.strip()
)


# =========================================================
# 5. Load AMRFinderPlus pilot reference
# =========================================================

df_amrfinder_pilot_hits = pd.read_csv(
    AMRFINDER_PILOT_HITS_PATH,
    dtype={
        "Genome ID": str
    }
)

df_amrfinder_pilot_metadata = pd.read_csv(
    AMRFINDER_PILOT_METADATA_PATH
)

df_amrfinder_strategy = pd.read_csv(
    AMRFINDER_STRATEGY_PATH
)


# =========================================================
# 6. Cohort integrity checks
# =========================================================

assert len(df_feature_input) == 4233, (
    f"Expected 4,233 genomes, "
    f"found {len(df_feature_input):,}."
)


assert (
    df_feature_input[
        "Genome ID"
    ]
    .nunique()
    == 4233
), "Duplicate Genome IDs detected."


assert (
    df_feature_input[
        "Complete FASTA"
    ]
    .eq(True)
    .all()
), "Incomplete FASTA detected."


assert (
    df_feature_input[
        "Resistant Phenotype"
    ]
    .isin(
        [
            "Resistant",
            "Susceptible"
        ]
    )
    .all()
), "Unexpected phenotype value detected."


# =========================================================
# 7. Final summary
# =========================================================

print("=" * 70)
print("FILE 04 — KNOWN AMR FEATURE ENGINEERING")
print("=" * 70)

print(
    f"Total genomes             : "
    f"{len(df_feature_input):,}"
)

print(
    f"Unique Genome IDs         : "
    f"{df_feature_input['Genome ID'].nunique():,}"
)

print(
    f"Validated FASTA genomes   : "
    f"{df_feature_input['Complete FASTA'].sum():,}"
)

print()

print("Phenotype distribution:")

print(
    df_feature_input[
        "Resistant Phenotype"
    ]
    .value_counts()
)

print()

print(
    f"Pilot AMRFinder hits      : "
    f"{len(df_amrfinder_pilot_hits):,}"
)

print(
    f"Pilot genomes represented : "
    f"{df_amrfinder_pilot_hits['Genome ID'].nunique():,}"
)

print()

print("AMRFinder pilot metadata:")
display(
    df_amrfinder_pilot_metadata
)

print()

print("File 03 screening decision:")
display(
    df_amrfinder_strategy
)

print()

print(
    "PASS: File 04 cohort and AMRFinderPlus "
    "reference data are loaded and ready."
)

Mounted at /content/drive
FILE 04 — KNOWN AMR FEATURE ENGINEERING
Total genomes             : 4,233
Unique Genome IDs         : 4,233
Validated FASTA genomes   : 4,233

Phenotype distribution:
Resistant Phenotype
Susceptible    2630
Resistant      1603
Name: count, dtype: int64

Pilot AMRFinder hits      : 104
Pilot genomes represented : 6

AMRFinder pilot metadata:


,AMRFinderPlus Version,Database Version,Organism,Pilot Genomes,Successful Runs
0,4.2.7,2026-08-07.1,Klebsiella_pneumoniae,6,6



File 03 screening decision:


,Reference Method,AMRFinderPlus Version,Database Version,Organism,Nucleotide Runtime Seconds per Genome,Combined Runtime Seconds per Genome,Protein Runtime Seconds per Genome,Combined Original-only AMR Occurrences,Combined-only AMR Occurrences,Protein Original-only AMR Occurrences,Protein-only AMR Occurrences,Protein Mutation Recall,Estimated Serial Full-Cohort Hours,Full Local Screening Status,Next Strategy
0,AMRFinderPlus nucleotide-only,4.2.7,2026-08-07.1,Klebsiella_pneumoniae,35.2,108.88,74.61,0,0,10,1,0.769,41.4,Deferred,Evaluate BV-BRC precomputed curated AMR annota...



PASS: File 04 cohort and AMRFinderPlus reference data are loaded and ready.


# Cell 2 — Pilot Retrieval of Precomputed BV-BRC AMR Annotations

In [ ]:
import os
import time
import requests
import pandas as pd


# =========================================================
# 1. BV-BRC API configuration
# =========================================================

BV_BRC_SP_GENE_API = (
    "https://www.bv-brc.org/api/sp_gene/"
)

session = requests.Session()

session.headers.update(
    {
        "Accept": "application/json",
        "User-Agent": (
            "AMR-Genome-ML/1.0 "
            "(academic research)"
        )
    }
)


# =========================================================
# 2. Recover the same six pilot Genome IDs
# =========================================================

pilot_genome_ids = (
    df_amrfinder_pilot_hits[
        "Genome ID"
    ]
    .astype(str)
    .drop_duplicates()
    .tolist()
)


assert len(pilot_genome_ids) == 6, (
    f"Expected 6 pilot genomes, "
    f"found {len(pilot_genome_ids)}."
)


print("Pilot Genome IDs:")

for genome_id in pilot_genome_ids:
    print(" -", genome_id)


# =========================================================
# 3. Retrieve precomputed BV-BRC specialty-gene records
# =========================================================

pilot_bvbrc_records = []

retrieval_audit = []


for i, genome_id in enumerate(
    pilot_genome_ids,
    start=1
):

    query_url = (
        f"{BV_BRC_SP_GENE_API}"
        f"?eq(genome_id,{genome_id})"
        f"&limit(10000)"
    )


    start_time = time.time()


    try:

        response = session.get(
            query_url,
            timeout=60
        )


        response.raise_for_status()


        records = response.json()


        elapsed = (
            time.time()
            - start_time
        )


        if not isinstance(
            records,
            list
        ):

            raise RuntimeError(
                "Unexpected BV-BRC API response format."
            )


        for record in records:

            record[
                "Pilot Genome ID"
            ] = genome_id

            pilot_bvbrc_records.append(
                record
            )


        retrieval_audit.append(
            {
                "Genome ID":
                    genome_id,

                "Status":
                    "PASS",

                "Records Retrieved":
                    len(records),

                "Runtime Seconds":
                    round(
                        elapsed,
                        2
                    ),

                "Error":
                    None
            }
        )


        print(
            f"[{i}/6] {genome_id} | "
            f"{len(records):,} records | "
            f"{elapsed:.2f} s"
        )


    except Exception as exc:

        retrieval_audit.append(
            {
                "Genome ID":
                    genome_id,

                "Status":
                    "FAIL",

                "Records Retrieved":
                    0,

                "Runtime Seconds":
                    None,

                "Error":
                    str(exc)
            }
        )


        print(
            f"[{i}/6] {genome_id} | FAIL"
        )


# =========================================================
# 4. Convert results to DataFrames
# =========================================================

df_bvbrc_pilot_raw = pd.DataFrame(
    pilot_bvbrc_records
)


df_bvbrc_pilot_audit = pd.DataFrame(
    retrieval_audit
)


# =========================================================
# 5. Basic retrieval integrity
# =========================================================

successful_runs = int(
    df_bvbrc_pilot_audit[
        "Status"
    ]
    .eq("PASS")
    .sum()
)


if successful_runs != 6:

    display(
        df_bvbrc_pilot_audit
    )

    raise RuntimeError(
        "One or more BV-BRC pilot "
        "retrieval requests failed."
    )


# =========================================================
# 6. Inspect available fields
# =========================================================

print()
print("=" * 70)
print("BV-BRC PRECOMPUTED ANNOTATION PILOT")
print("=" * 70)

print(
    f"Pilot genomes requested : "
    f"{len(pilot_genome_ids)}"
)

print(
    f"Successful requests     : "
    f"{successful_runs}/6"
)

print(
    f"Total records retrieved : "
    f"{len(df_bvbrc_pilot_raw):,}"
)


if not df_bvbrc_pilot_raw.empty:

    print(
        f"Genomes represented     : "
        f"{df_bvbrc_pilot_raw['genome_id'].astype(str).nunique():,}"
    )

    print()

    print("Returned columns:")

    print(
        df_bvbrc_pilot_raw
        .columns
        .tolist()
    )


# =========================================================
# 7. Inspect property distribution
# =========================================================

if (
    not df_bvbrc_pilot_raw.empty
    and "property"
    in df_bvbrc_pilot_raw.columns
):

    print()
    print("BV-BRC property distribution:")

    print(
        df_bvbrc_pilot_raw[
            "property"
        ]
        .value_counts(
            dropna=False
        )
    )


# =========================================================
# 8. Extract AMR records only
# =========================================================

if (
    not df_bvbrc_pilot_raw.empty
    and "property"
    in df_bvbrc_pilot_raw.columns
):

    df_bvbrc_pilot_amr = (
        df_bvbrc_pilot_raw[
            df_bvbrc_pilot_raw[
                "property"
            ]
            .astype(str)
            .str.upper()
            .eq("AMR")
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

else:

    df_bvbrc_pilot_amr = pd.DataFrame()


# =========================================================
# 9. AMR summary
# =========================================================

print()
print(
    f"AMR records retained    : "
    f"{len(df_bvbrc_pilot_amr):,}"
)


if not df_bvbrc_pilot_amr.empty:

    print(
        f"AMR genomes represented : "
        f"{df_bvbrc_pilot_amr['genome_id'].astype(str).nunique():,}"
    )


    if "gene" in df_bvbrc_pilot_amr.columns:

        print(
            f"Unique AMR gene labels  : "
            f"{df_bvbrc_pilot_amr['gene'].dropna().astype(str).nunique():,}"
        )


# =========================================================
# 10. Preview useful columns
# =========================================================

preview_columns = [
    "genome_id",
    "gene",
    "function",
    "property",
    "property_source",
    "source",
    "source_id",
    "antibiotics",
    "antibiotics_class",
    "classification",
    "identity",
    "query_coverage",
    "subject_coverage"
]


preview_columns = [
    col
    for col in preview_columns
    if col in df_bvbrc_pilot_amr.columns
]


if not df_bvbrc_pilot_amr.empty:

    print()
    print("Example BV-BRC AMR annotations:")

    display(
        df_bvbrc_pilot_amr[
            preview_columns
        ]
        .head(20)
    )


# =========================================================
# 11. Save pilot retrieval to Drive
# =========================================================

BV_BRC_PILOT_RAW_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "bvbrc_pilot_specialty_genes_raw.csv"
)

BV_BRC_PILOT_AMR_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "bvbrc_pilot_amr_annotations.csv"
)

BV_BRC_PILOT_AUDIT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "bvbrc_pilot_retrieval_audit.csv"
)


df_bvbrc_pilot_raw.to_csv(
    BV_BRC_PILOT_RAW_PATH,
    index=False
)

df_bvbrc_pilot_amr.to_csv(
    BV_BRC_PILOT_AMR_PATH,
    index=False
)

df_bvbrc_pilot_audit.to_csv(
    BV_BRC_PILOT_AUDIT_PATH,
    index=False
)


print()
print(
    "PASS: BV-BRC precomputed annotations "
    "were retrieved for all six pilot genomes."
)

Pilot Genome IDs:
 - 1284787.3
 - 1284788.3
 - 1284789.4
 - 1284805.4
 - 1284814.3
 - 1284818.4
[1/6] 1284787.3 | 2,255 records | 7.11 s
[2/6] 1284788.3 | 2,271 records | 4.34 s
[3/6] 1284789.4 | 2,243 records | 6.12 s
[4/6] 1284805.4 | 2,283 records | 3.48 s
[5/6] 1284814.3 | 2,260 records | 2.72 s
[6/6] 1284818.4 | 2,329 records | 4.23 s

BV-BRC PRECOMPUTED ANNOTATION PILOT
Pilot genomes requested : 6
Successful requests     : 6/6
Total records retrieved : 13,641
Genomes represented     : 6

Returned columns:
['alt_locus_tag', 'antibiotics_class', 'date_inserted', 'date_modified', 'e_value', 'evidence', 'feature_id', 'function', 'genome_id', 'genome_name', 'id', 'identity', 'organism', 'owner', 'patric_id', 'product', 'property', 'property_source', 'public', 'query_coverage', 'refseq_locus_tag', 'source', 'source_id', 'subject_coverage', 'taxon_id', 'classification', 'pmid', '_version_', 'Pilot Genome ID', 'gene', 'same_genome', 'same_genus', 'same_species', 'antibiotics', 'assertion

# Cell 3 — Extract BV-BRC Antibiotic Resistance Annotations

In [ ]:
# =========================================================
# 1. Correct BV-BRC AMR filtering rule
# =========================================================

df_bvbrc_pilot_amr = (
    df_bvbrc_pilot_raw[
        df_bvbrc_pilot_raw[
            "property"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            "antibiotic resistance"
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# =========================================================
# 2. Basic integrity checks
# =========================================================

assert len(df_bvbrc_pilot_amr) > 0, (
    "No BV-BRC antibiotic-resistance "
    "annotations were detected."
)


assert (
    df_bvbrc_pilot_amr[
        "genome_id"
    ]
    .astype(str)
    .nunique()
    == 6
), (
    "Not all six pilot genomes are represented."
)


# =========================================================
# 3. Basic summary
# =========================================================

print("=" * 70)
print("BV-BRC ANTIBIOTIC RESISTANCE ANNOTATION SUMMARY")
print("=" * 70)

print(
    f"AMR records retained       : "
    f"{len(df_bvbrc_pilot_amr):,}"
)

print(
    f"Genomes represented        : "
    f"{df_bvbrc_pilot_amr['genome_id'].astype(str).nunique():,}"
)


# =========================================================
# 4. Records per genome
# =========================================================

print()
print("AMR records per genome:")

print(
    df_bvbrc_pilot_amr[
        "genome_id"
    ]
    .astype(str)
    .value_counts()
    .sort_index()
)


# =========================================================
# 5. Annotation-source distribution
# =========================================================

if (
    "property_source"
    in df_bvbrc_pilot_amr.columns
):

    print()
    print("Property source distribution:")

    print(
        df_bvbrc_pilot_amr[
            "property_source"
        ]
        .value_counts(
            dropna=False
        )
        .head(30)
    )


if (
    "source"
    in df_bvbrc_pilot_amr.columns
):

    print()
    print("Source distribution:")

    print(
        df_bvbrc_pilot_amr[
            "source"
        ]
        .value_counts(
            dropna=False
        )
        .head(30)
    )


# =========================================================
# 6. Resistance-class distribution
# =========================================================

if (
    "antibiotics_class"
    in df_bvbrc_pilot_amr.columns
):

    print()
    print("Antibiotic class distribution:")

    print(
        df_bvbrc_pilot_amr[
            "antibiotics_class"
        ]
        .value_counts(
            dropna=False
        )
        .head(30)
    )


# =========================================================
# 7. Gene-label audit
# =========================================================

non_null_gene_count = (
    df_bvbrc_pilot_amr[
        "gene"
    ]
    .notna()
    .sum()
)

unique_gene_count = (
    df_bvbrc_pilot_amr[
        "gene"
    ]
    .dropna()
    .astype(str)
    .nunique()
)


print()
print(
    f"Records with gene label    : "
    f"{non_null_gene_count:,}"
)

print(
    f"Unique gene labels         : "
    f"{unique_gene_count:,}"
)


# =========================================================
# 8. Preview useful columns
# =========================================================

preview_columns = [
    "genome_id",
    "gene",
    "product",
    "function",
    "property_source",
    "source",
    "source_id",
    "antibiotics",
    "antibiotics_class",
    "classification",
    "evidence",
    "identity",
    "query_coverage",
    "subject_coverage"
]


preview_columns = [
    col
    for col in preview_columns
    if col in df_bvbrc_pilot_amr.columns
]


print()
print("Example BV-BRC resistance annotations:")

display(
    df_bvbrc_pilot_amr[
        preview_columns
    ]
    .head(30)
)


# =========================================================
# 9. Save corrected pilot AMR table
# =========================================================

BV_BRC_PILOT_AMR_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "bvbrc_pilot_amr_annotations.csv"
)


df_bvbrc_pilot_amr.to_csv(
    BV_BRC_PILOT_AMR_PATH,
    index=False
)


print()
print(
    "Saved corrected BV-BRC AMR annotations to:"
)

print(
    BV_BRC_PILOT_AMR_PATH
)

print()

print(
    "PASS: BV-BRC antibiotic-resistance annotations "
    "were successfully extracted for all six pilot genomes."
)

BV-BRC ANTIBIOTIC RESISTANCE ANNOTATION SUMMARY
AMR records retained       : 1,093
Genomes represented        : 6

AMR records per genome:
genome_id
1284787.3    182
1284788.3    179
1284789.4    168
1284805.4    193
1284814.3    176
1284818.4    195
Name: count, dtype: int64

Property source distribution:
property_source
Antibiotic Resistance: CARD      592
Antibiotic Resistance: PATRIC    452
Antibiotic Resistance: NDARO      49
Name: count, dtype: int64

Source distribution:
source
CARD      592
PATRIC    452
NDARO      49
Name: count, dtype: int64

Antibiotic class distribution:
antibiotics_class
                                                                                                                                                                                  404
polymixins                                                                                                                                                                         70
aminoglycosides             

,genome_id,gene,product,function,property_source,source,source_id,antibiotics,antibiotics_class,classification,evidence,identity,query_coverage,subject_coverage
0,1284787.3,NaN,"Multidrug efflux system MdtABC-TolC, inner-mem...",NaN,Antibiotic Resistance: PATRIC,PATRIC,,[novobiocin],aminocoumarin antibiotics,[efflux pump conferring antibiotic resistance],K-mer Search,NaN,NaN,NaN
1,1284787.3,folA,Dihydrofolate reductase (EC 1.5.1.3),NaN,Antibiotic Resistance: PATRIC,PATRIC,,"[trimethoprim, brodimoprim, tetroxoprim, iclap...",diaminopyrimidines,[antibiotic target in susceptible species],K-mer Search,NaN,NaN,NaN
2,1284787.3,frp,Oxygen-insensitive NADPH nitroreductase (EC 1....,,Antibiotic Resistance: CARD,CARD,NP_415372.1,NaN,,"[antibiotic resistant gene variant or mutant, ...",BLAT,85.0,99.0,45.0
3,1284787.3,gyrA,DNA gyrase subunit A (EC 5.99.1.3),,Antibiotic Resistance: CARD,CARD,NP_461214.1,NaN,,"[antibiotic resistant gene variant or mutant, ...",BLAT,95.0,99.0,99.0
4,1284787.3,yojI,"ABC-type siderophore export system, fused ATPa...",,Antibiotic Resistance: CARD,CARD,NP_416715.1,[microcin J25],,[efflux pump conferring antibiotic resistance],BLAT,83.0,100.0,100.0
5,1284787.3,ramA_1,Transcriptional activator RamA,,Antibiotic Resistance: CARD,CARD,AFK13828.1,NaN,,"[efflux pump conferring antibiotic resistance,...",BLAT,92.0,100.0,91.0
6,1284787.3,dxr,1-deoxy-D-xylulose 5-phosphate reductoisomeras...,NaN,Antibiotic Resistance: PATRIC,PATRIC,,[fosmidomycin],fosmidomycin,[antibiotic target in susceptible species],K-mer Search,NaN,NaN,NaN
7,1284787.3,rpoC,DNA-directed RNA polymerase beta' subunit (EC ...,NaN,Antibiotic Resistance: PATRIC,PATRIC,,[daptomycin],"myxopyronins corallopyronins,peptide antibiotics",[antibiotic target in susceptible species],K-mer Search,NaN,NaN,NaN
8,1284787.3,marR,Multiple antibiotic resistance protein MarR,NaN,Antibiotic Resistance: PATRIC,PATRIC,,"[cefalothin, chloramphenicol, tigecycline, amp...","penams,carbapenems,cephamycins,cephalosporins,...","[antibiotic resistance gene cluster, cassette,...",K-mer Search,NaN,NaN,NaN
9,1284787.3,gyrB,DNA gyrase subunit B (EC 5.99.1.3),,Antibiotic Resistance: CARD,CARD,BAE77595.1,NaN,,"[aminocoumarin resistance gene, antibiotic res...",BLAT,95.0,100.0,100.0



Saved corrected BV-BRC AMR annotations to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/bvbrc_pilot_amr_annotations.csv

PASS: BV-BRC antibiotic-resistance annotations were successfully extracted for all six pilot genomes.


# Cell 4 — Compare BV-BRC Annotations with AMRFinderPlus Pilot Results

In [ ]:
import re
import os
import pandas as pd


# =========================================================
# 1. Prepare AMRFinder AMR-only reference
# =========================================================

df_amrfinder_reference = (
    df_amrfinder_pilot_hits[
        df_amrfinder_pilot_hits[
            "Type"
        ]
        .astype(str)
        .str.upper()
        .eq("AMR")
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(df_amrfinder_reference) == 99, (
    f"Expected 99 AMRFinder AMR hits, "
    f"found {len(df_amrfinder_reference)}."
)


# =========================================================
# 2. Label normalization
# =========================================================

def normalize_label(value):

    if pd.isna(value):
        return None

    value = str(value).strip()

    if not value:
        return None

    return re.sub(
        r"[^a-z0-9]",
        "",
        value.lower()
    )


# =========================================================
# 3. Convert determinant name to underlying gene
# =========================================================

def determinant_to_gene(value):

    if pd.isna(value):
        return None

    value = str(value).strip()

    if not value:
        return None


    # ---------------------------------------------
    # Remove explicit mutation suffixes
    #
    # gyrA_S83I       -> gyrA
    # ompK35_E42...   -> ompK35
    # mgrB_C28Y       -> mgrB
    # ---------------------------------------------

    if re.search(
        r"_[A-Za-z]\d",
        value
    ):

        value = value.split(
            "_",
            1
        )[0]


    # ---------------------------------------------
    # Beta-lactamase allele numbers
    #
    # blaKPC-3   -> blaKPC
    # blaSHV-12  -> blaSHV
    # blaOXA-9   -> blaOXA
    # ---------------------------------------------

    if value.lower().startswith("bla"):

        value = re.sub(
            r"-\d+[A-Za-z]*$",
            "",
            value
        )


    # ---------------------------------------------
    # Remove BV-BRC duplicate suffixes
    #
    # ramA_1 -> ramA
    # folP_2 -> folP
    # ---------------------------------------------

    value = re.sub(
        r"_\d+$",
        "",
        value
    )


    return normalize_label(
        value
    )


# =========================================================
# 4. Build usable BV-BRC label table
# =========================================================

bvbrc_label_records = []


for _, row in df_bvbrc_pilot_amr.iterrows():

    genome_id = str(
        row["genome_id"]
    ).strip()

    source = str(
        row.get(
            "source",
            ""
        )
    ).strip()


    # ---------------------------------------------
    # Primary gene field
    # ---------------------------------------------

    gene_value = row.get(
        "gene"
    )


    if (
        pd.notna(gene_value)
        and str(gene_value).strip()
    ):

        label = str(
            gene_value
        ).strip()

        bvbrc_label_records.append(
            {
                "Genome ID":
                    genome_id,

                "BV-BRC Label":
                    label,

                "Label Field":
                    "gene",

                "Source":
                    source
            }
        )


    # ---------------------------------------------
    # Some CARD records store the gene name in
    # the function field instead of gene.
    #
    # Only accept short single-token values here.
    # ---------------------------------------------

    function_value = row.get(
        "function"
    )


    if (
        pd.notna(function_value)
        and str(function_value).strip()
    ):

        function_value = str(
            function_value
        ).strip()


        if re.fullmatch(
            r"[A-Za-z][A-Za-z0-9()'._+\-/]{1,39}",
            function_value
        ):

            bvbrc_label_records.append(
                {
                    "Genome ID":
                        genome_id,

                    "BV-BRC Label":
                        function_value,

                    "Label Field":
                        "function",

                    "Source":
                        source
                }
            )


df_bvbrc_labels = (
    pd.DataFrame(
        bvbrc_label_records
    )
    .drop_duplicates()
    .reset_index(drop=True)
)


# =========================================================
# 5. Add normalized labels
# =========================================================

df_bvbrc_labels[
    "Exact Normalized"
] = (
    df_bvbrc_labels[
        "BV-BRC Label"
    ]
    .apply(
        normalize_label
    )
)


df_bvbrc_labels[
    "Gene Normalized"
] = (
    df_bvbrc_labels[
        "BV-BRC Label"
    ]
    .apply(
        determinant_to_gene
    )
)


# =========================================================
# 6. Prepare AMRFinder determinant table
# =========================================================

df_amrfinder_compare = (
    df_amrfinder_reference[
        [
            "Genome ID",
            "Element symbol",
            "Subtype",
            "Class"
        ]
    ]
    .copy()
)


df_amrfinder_compare[
    "Genome ID"
] = (
    df_amrfinder_compare[
        "Genome ID"
    ]
    .astype(str)
)


df_amrfinder_compare[
    "Exact Normalized"
] = (
    df_amrfinder_compare[
        "Element symbol"
    ]
    .apply(
        normalize_label
    )
)


df_amrfinder_compare[
    "Gene Normalized"
] = (
    df_amrfinder_compare[
        "Element symbol"
    ]
    .apply(
        determinant_to_gene
    )
)


# =========================================================
# 7. Compare each AMRFinder determinant with BV-BRC
# =========================================================

comparison_records = []


for _, row in df_amrfinder_compare.iterrows():

    genome_id = row[
        "Genome ID"
    ]

    exact_label = row[
        "Exact Normalized"
    ]

    gene_label = row[
        "Gene Normalized"
    ]


    genome_bvbrc = (
        df_bvbrc_labels[
            df_bvbrc_labels[
                "Genome ID"
            ] == genome_id
        ]
    )


    exact_matches = (
        genome_bvbrc[
            genome_bvbrc[
                "Exact Normalized"
            ] == exact_label
        ]
    )


    gene_matches = (
        genome_bvbrc[
            genome_bvbrc[
                "Gene Normalized"
            ] == gene_label
        ]
    )


    comparison_records.append(
        {
            "Genome ID":
                genome_id,

            "AMRFinder Determinant":
                row[
                    "Element symbol"
                ],

            "Subtype":
                row[
                    "Subtype"
                ],

            "Class":
                row[
                    "Class"
                ],

            "Exact Match":
                len(
                    exact_matches
                ) > 0,

            "Gene-level Match":
                len(
                    gene_matches
                ) > 0,

            "Matched BV-BRC Labels":
                ";".join(
                    sorted(
                        gene_matches[
                            "BV-BRC Label"
                        ]
                        .astype(str)
                        .unique()
                    )
                ),

            "Matched Sources":
                ";".join(
                    sorted(
                        gene_matches[
                            "Source"
                        ]
                        .astype(str)
                        .unique()
                    )
                )
        }
    )


df_amrfinder_vs_bvbrc = pd.DataFrame(
    comparison_records
)


# =========================================================
# 8. Overall coverage
# =========================================================

total_reference = len(
    df_amrfinder_vs_bvbrc
)


exact_matches = int(
    df_amrfinder_vs_bvbrc[
        "Exact Match"
    ]
    .sum()
)


gene_matches = int(
    df_amrfinder_vs_bvbrc[
        "Gene-level Match"
    ]
    .sum()
)


exact_coverage = (
    exact_matches
    / total_reference
)


gene_coverage = (
    gene_matches
    / total_reference
)


# =========================================================
# 9. Mutation-specific coverage
# =========================================================

mutation_mask = (
    df_amrfinder_vs_bvbrc[
        "Subtype"
    ]
    .isin(
        [
            "POINT",
            "POINT_DISRUPT"
        ]
    )
)


df_mutation_comparison = (
    df_amrfinder_vs_bvbrc[
        mutation_mask
    ]
)


mutation_total = len(
    df_mutation_comparison
)


mutation_exact = int(
    df_mutation_comparison[
        "Exact Match"
    ]
    .sum()
)


mutation_gene = int(
    df_mutation_comparison[
        "Gene-level Match"
    ]
    .sum()
)


# =========================================================
# 10. Save comparison
# =========================================================

BV_BRC_COMPARISON_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "amrfinder_vs_bvbrc_pilot_comparison.csv"
)


df_amrfinder_vs_bvbrc.to_csv(
    BV_BRC_COMPARISON_PATH,
    index=False
)


# =========================================================
# 11. Summary
# =========================================================

print("=" * 70)
print("AMRFINDERPLUS vs BV-BRC PILOT COMPARISON")
print("=" * 70)

print(
    f"AMRFinder AMR occurrences : "
    f"{total_reference}"
)

print()

print(
    f"Exact-label matches        : "
    f"{exact_matches}/{total_reference} "
    f"({exact_coverage:.1%})"
)

print(
    f"Gene-level matches         : "
    f"{gene_matches}/{total_reference} "
    f"({gene_coverage:.1%})"
)


print()
print("Mutation-specific comparison:")

print(
    f"Mutation occurrences       : "
    f"{mutation_total}"
)

print(
    f"Exact mutation matches     : "
    f"{mutation_exact}/{mutation_total}"
)

print(
    f"Underlying gene matches    : "
    f"{mutation_gene}/{mutation_total}"
)


# =========================================================
# 12. Missing AMRFinder determinants
# =========================================================

df_missing_bvbrc = (
    df_amrfinder_vs_bvbrc[
        ~df_amrfinder_vs_bvbrc[
            "Gene-level Match"
        ]
    ]
    .copy()
)


print()
print(
    f"AMRFinder determinants with "
    f"no BV-BRC gene-level match: "
    f"{len(df_missing_bvbrc)}"
)


if not df_missing_bvbrc.empty:

    display(
        df_missing_bvbrc[
            [
                "Genome ID",
                "AMRFinder Determinant",
                "Subtype",
                "Class"
            ]
        ]
    )


# =========================================================
# 13. Example successful matches
# =========================================================

print()
print("Example successful matches:")

display(
    df_amrfinder_vs_bvbrc[
        df_amrfinder_vs_bvbrc[
            "Gene-level Match"
        ]
    ]
    .head(30)
)


print()
print(
    "Comparison saved to:"
)

print(
    BV_BRC_COMPARISON_PATH
)

print()

print(
    "PASS: BV-BRC annotations were compared "
    "against the AMRFinderPlus pilot reference."
)

AMRFINDERPLUS vs BV-BRC PILOT COMPARISON
AMRFinder AMR occurrences : 99

Exact-label matches        : 35/99 (35.4%)
Gene-level matches         : 50/99 (50.5%)

Mutation-specific comparison:
Mutation occurrences       : 26
Exact mutation matches     : 0/26
Underlying gene matches    : 15/26

AMRFinder determinants with no BV-BRC gene-level match: 49


,Genome ID,AMRFinder Determinant,Subtype,Class
4,1284787.3,ompK35_E42RfsTer47,POINT_DISRUPT,BETA-LACTAM
5,1284787.3,aac(3)-IVa,AMR,AMINOGLYCOSIDE
7,1284787.3,blaSHV-12,AMR,BETA-LACTAM
9,1284787.3,blaKPC-3,AMR,BETA-LACTAM
12,1284787.3,blaOXA-9,AMR,BETA-LACTAM
13,1284787.3,estX-3,AMR,MACROLIDE
14,1284787.3,aadA2,AMR,AMINOGLYCOSIDE
16,1284788.3,dfrA12,AMR,TRIMETHOPRIM
20,1284788.3,blaOXA,AMR,BETA-LACTAM
21,1284788.3,ompK35_E42RfsTer47,POINT_DISRUPT,BETA-LACTAM



Example successful matches:


,Genome ID,AMRFinder Determinant,Subtype,Class,Exact Match,Gene-level Match,Matched BV-BRC Labels,Matched Sources
0,1284787.3,gyrA_D87N,POINT,QUINOLONE,False,True,gyrA,CARD;PATRIC
1,1284787.3,gyrA_S83I,POINT,QUINOLONE,False,True,gyrA,CARD;PATRIC
2,1284787.3,fosA,AMR,FOSFOMYCIN,True,True,fosA,CARD;NDARO
3,1284787.3,cmlA1,AMR,PHENICOL,True,True,cmlA1,CARD
6,1284787.3,aph(4)-Ia,AMR,AMINOGLYCOSIDE,True,True,APH(4)-Ia,CARD
8,1284787.3,sul3,AMR,SULFONAMIDE,True,True,sul3,CARD
10,1284787.3,oqxA,AMR,NITROFURAN/PHENICOL/QUINOLONE/TETRACYCLINE,True,True,oqxA,CARD
11,1284787.3,oqxB,AMR,NITROFURAN/PHENICOL/QUINOLONE/TETRACYCLINE,True,True,oqxB,CARD
15,1284787.3,parC_S80I,POINT,QUINOLONE,False,True,parC,CARD
17,1284788.3,aadA2,AMR,AMINOGLYCOSIDE,True,True,aadA2,CARD



Comparison saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/amrfinder_vs_bvbrc_pilot_comparison.csv

PASS: BV-BRC annotations were compared against the AMRFinderPlus pilot reference.


# Cell 5 — Deep Audit of Apparently Missing BV-BRC Determinants

In [ ]:
import os
import re
import pandas as pd


# =========================================================
# 1. AMRFinder determinants currently considered missing
# =========================================================

df_missing_candidates = (
    df_amrfinder_vs_bvbrc[
        ~df_amrfinder_vs_bvbrc[
            "Gene-level Match"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(df_missing_candidates) == 49


# =========================================================
# 2. BV-BRC fields to search
# =========================================================

SEARCH_FIELDS = [
    "gene",
    "function",
    "product",
    "source_id",
    "classification",
    "antibiotics",
    "antibiotics_class",
    "assertion"
]


SEARCH_FIELDS = [
    col
    for col in SEARCH_FIELDS
    if col in df_bvbrc_pilot_amr.columns
]


# =========================================================
# 3. General normalization
# =========================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower()
    )


def base_gene_name(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()


    # Mutation suffix
    if re.search(
        r"_[A-Za-z]\d",
        value
    ):

        value = value.split(
            "_",
            1
        )[0]


    # blaKPC-3 -> blaKPC
    if value.lower().startswith("bla"):

        value = re.sub(
            r"-\d+[A-Za-z]*$",
            "",
            value
        )


    return normalize_text(
        value
    )


# =========================================================
# 4. Build searchable text per BV-BRC record
# =========================================================

df_bvbrc_search = (
    df_bvbrc_pilot_amr
    .copy()
)


df_bvbrc_search[
    "Search Text"
] = (
    df_bvbrc_search[
        SEARCH_FIELDS
    ]
    .fillna("")
    .astype(str)
    .agg(
        " | ".join,
        axis=1
    )
)


df_bvbrc_search[
    "Search Normalized"
] = (
    df_bvbrc_search[
        "Search Text"
    ]
    .apply(
        normalize_text
    )
)


# =========================================================
# 5. Deep-search each missing AMRFinder determinant
# =========================================================

deep_audit_records = []


for _, row in df_missing_candidates.iterrows():

    genome_id = str(
        row["Genome ID"]
    )

    determinant = str(
        row["AMRFinder Determinant"]
    )


    exact_token = normalize_text(
        determinant
    )


    gene_token = base_gene_name(
        determinant
    )


    genome_records = (
        df_bvbrc_search[
            df_bvbrc_search[
                "genome_id"
            ]
            .astype(str)
            .eq(genome_id)
        ]
    )


    # ---------------------------------------------
    # Exact determinant string anywhere
    # ---------------------------------------------

    exact_hits = (
        genome_records[
            genome_records[
                "Search Normalized"
            ]
            .str.contains(
                exact_token,
                regex=False
            )
        ]
        if exact_token
        else genome_records.iloc[0:0]
    )


    # ---------------------------------------------
    # Underlying gene string anywhere
    # ---------------------------------------------

    gene_hits = (
        genome_records[
            genome_records[
                "Search Normalized"
            ]
            .str.contains(
                gene_token,
                regex=False
            )
        ]
        if gene_token
        else genome_records.iloc[0:0]
    )


    recovered = (
        len(
            exact_hits
        ) > 0
        or
        len(
            gene_hits
        ) > 0
    )


    matched_records = (
        exact_hits
        if len(exact_hits) > 0
        else gene_hits
    )


    matched_sources = ""

    matched_text_examples = ""


    if not matched_records.empty:

        if "source" in matched_records.columns:

            matched_sources = ";".join(
                sorted(
                    matched_records[
                        "source"
                    ]
                    .dropna()
                    .astype(str)
                    .unique()
                )
            )


        matched_text_examples = " || ".join(
            matched_records[
                "Search Text"
            ]
            .astype(str)
            .drop_duplicates()
            .head(3)
            .tolist()
        )


    deep_audit_records.append(
        {
            "Genome ID":
                genome_id,

            "AMRFinder Determinant":
                determinant,

            "Subtype":
                row[
                    "Subtype"
                ],

            "Class":
                row[
                    "Class"
                ],

            "Exact Text Recovered":
                len(
                    exact_hits
                ) > 0,

            "Gene Text Recovered":
                len(
                    gene_hits
                ) > 0,

            "Recovered Anywhere":
                recovered,

            "Matched Sources":
                matched_sources,

            "Matched BV-BRC Text":
                matched_text_examples
        }
    )


df_bvbrc_deep_audit = pd.DataFrame(
    deep_audit_records
)


# =========================================================
# 6. Summary
# =========================================================

recovered_count = int(
    df_bvbrc_deep_audit[
        "Recovered Anywhere"
    ]
    .sum()
)


still_missing_count = (
    len(
        df_bvbrc_deep_audit
    )
    - recovered_count
)


print("=" * 70)
print("DEEP BV-BRC DETERMINANT AUDIT")
print("=" * 70)

print(
    f"Previously missing determinants : "
    f"{len(df_bvbrc_deep_audit)}"
)

print(
    f"Recovered in other fields       : "
    f"{recovered_count}"
)

print(
    f"Still not detected              : "
    f"{still_missing_count}"
)


print()
print(
    "Recovery by subtype:"
)

print(
    df_bvbrc_deep_audit
    .groupby(
        "Subtype"
    )[
        "Recovered Anywhere"
    ]
    .agg(
        [
            "count",
            "sum"
        ]
    )
)


# =========================================================
# 7. Still-missing determinants
# =========================================================

df_still_missing = (
    df_bvbrc_deep_audit[
        ~df_bvbrc_deep_audit[
            "Recovered Anywhere"
        ]
    ]
    .copy()
)


print()
print("Determinants still missing after deep search:")

display(
    df_still_missing[
        [
            "Genome ID",
            "AMRFinder Determinant",
            "Subtype",
            "Class"
        ]
    ]
)


# =========================================================
# 8. Recovered examples
# =========================================================

df_recovered = (
    df_bvbrc_deep_audit[
        df_bvbrc_deep_audit[
            "Recovered Anywhere"
        ]
    ]
    .copy()
)


print()
print("Examples recovered from additional BV-BRC fields:")

display(
    df_recovered[
        [
            "Genome ID",
            "AMRFinder Determinant",
            "Subtype",
            "Class",
            "Exact Text Recovered",
            "Gene Text Recovered",
            "Matched Sources",
            "Matched BV-BRC Text"
        ]
    ]
    .head(30)
)


# =========================================================
# 9. Save deep audit
# =========================================================

BV_BRC_DEEP_AUDIT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "amrfinder_vs_bvbrc_deep_audit.csv"
)


df_bvbrc_deep_audit.to_csv(
    BV_BRC_DEEP_AUDIT_PATH,
    index=False
)


print()
print(
    "Deep audit saved to:"
)

print(
    BV_BRC_DEEP_AUDIT_PATH
)

print()

print(
    "PASS: Apparently missing AMRFinder determinants "
    "were searched across extended BV-BRC annotation fields."
)

DEEP BV-BRC DETERMINANT AUDIT
Previously missing determinants : 49
Recovered in other fields       : 4
Still not detected              : 45

Recovery by subtype:
               count  sum
Subtype                  
AMR               38    2
POINT              6    2
POINT_DISRUPT      5    0

Determinants still missing after deep search:


,Genome ID,AMRFinder Determinant,Subtype,Class
0,1284787.3,ompK35_E42RfsTer47,POINT_DISRUPT,BETA-LACTAM
1,1284787.3,aac(3)-IVa,AMR,AMINOGLYCOSIDE
2,1284787.3,blaSHV-12,AMR,BETA-LACTAM
3,1284787.3,blaKPC-3,AMR,BETA-LACTAM
4,1284787.3,blaOXA-9,AMR,BETA-LACTAM
5,1284787.3,estX-3,AMR,MACROLIDE
6,1284787.3,aadA2,AMR,AMINOGLYCOSIDE
7,1284788.3,dfrA12,AMR,TRIMETHOPRIM
8,1284788.3,blaOXA,AMR,BETA-LACTAM
9,1284788.3,ompK35_E42RfsTer47,POINT_DISRUPT,BETA-LACTAM



Examples recovered from additional BV-BRC fields:


,Genome ID,AMRFinder Determinant,Subtype,Class,Exact Text Recovered,Gene Text Recovered,Matched Sources,Matched BV-BRC Text
43,1284818.4,blaOXA-1,AMR,BETA-LACTAM,True,True,CARD,bla | OXA-1 | Class D beta-lactamase (EC 3.5.2...
45,1284818.4,mgrB_C28Y,POINT,COLISTIN,False,True,PATRIC,| | PhoP/PhoQ regulator MgrB | | ['gene con...
47,1284818.4,fosA,AMR,FOSFOMYCIN,True,True,CARD;NDARO,| | Fosfomycin resistance protein FosA | AJE...
48,1284818.4,ramR_A19V,POINT,TETRACYCLINE,False,True,CARD,| Klebsiella pneumoniae ramR mutants | Transc...



Deep audit saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/amrfinder_vs_bvbrc_deep_audit.csv

PASS: Apparently missing AMRFinder determinants were searched across extended BV-BRC annotation fields.


# Cell 6 — Map Pilot Genomes to Public NCBI Assembly Accessions

In [ ]:
import os
import time
import requests
import pandas as pd


# =========================================================
# 1. BV-BRC genome metadata API
# =========================================================

BV_BRC_GENOME_API = (
    "https://www.bv-brc.org/api/genome/"
)


metadata_session = requests.Session()

metadata_session.headers.update(
    {
        "Accept": "application/json",
        "User-Agent": (
            "AMR-Genome-ML/1.0 "
            "(academic research)"
        )
    }
)


# =========================================================
# 2. Recover the six pilot Genome IDs
# =========================================================

pilot_genome_ids = (
    df_amrfinder_pilot_hits[
        "Genome ID"
    ]
    .astype(str)
    .drop_duplicates()
    .tolist()
)


assert len(pilot_genome_ids) == 6


# =========================================================
# 3. Retrieve public-accession metadata
# =========================================================

assembly_records = []


for i, genome_id in enumerate(
    pilot_genome_ids,
    start=1
):

    query_url = (
        f"{BV_BRC_GENOME_API}"
        f"?eq(genome_id,{genome_id})"
        f"&limit(1)"
    )


    start_time = time.time()


    try:

        response = metadata_session.get(
            query_url,
            timeout=60
        )

        response.raise_for_status()

        records = response.json()


        if (
            not isinstance(records, list)
            or len(records) == 0
        ):

            raise RuntimeError(
                "No genome metadata returned."
            )


        record = records[0]


        assembly_records.append(
            {
                "Genome ID":
                    genome_id,

                "Genome Name":
                    record.get(
                        "genome_name"
                    ),

                "Assembly Accession":
                    record.get(
                        "assembly_accession"
                    ),

                "BioSample Accession":
                    record.get(
                        "biosample_accession"
                    ),

                "SRA Accession":
                    record.get(
                        "sra_accession"
                    ),

                "GenBank Accessions":
                    record.get(
                        "genbank_accessions"
                    ),

                "RefSeq Accessions":
                    record.get(
                        "refseq_accessions"
                    ),

                "Status":
                    "PASS",

                "Runtime Seconds":
                    round(
                        time.time()
                        - start_time,
                        2
                    )
            }
        )


        print(
            f"[{i}/6] {genome_id} | "
            f"Assembly: "
            f"{record.get('assembly_accession')}"
        )


    except Exception as exc:

        assembly_records.append(
            {
                "Genome ID":
                    genome_id,

                "Genome Name":
                    None,

                "Assembly Accession":
                    None,

                "BioSample Accession":
                    None,

                "SRA Accession":
                    None,

                "GenBank Accessions":
                    None,

                "RefSeq Accessions":
                    None,

                "Status":
                    "FAIL",

                "Runtime Seconds":
                    None,

                "Error":
                    str(exc)
            }
        )


# =========================================================
# 4. Build mapping table
# =========================================================

df_pilot_ncbi_mapping = pd.DataFrame(
    assembly_records
)


# =========================================================
# 5. Mapping summary
# =========================================================

assembly_available = int(
    df_pilot_ncbi_mapping[
        "Assembly Accession"
    ]
    .notna()
    .sum()
)


biosample_available = int(
    df_pilot_ncbi_mapping[
        "BioSample Accession"
    ]
    .notna()
    .sum()
)


print()
print("=" * 70)
print("PILOT GENOME → NCBI ACCESSION MAPPING")
print("=" * 70)

print(
    f"Pilot genomes             : "
    f"{len(df_pilot_ncbi_mapping)}"
)

print(
    f"Public assembly accessions: "
    f"{assembly_available}/6"
)

print(
    f"BioSample accessions      : "
    f"{biosample_available}/6"
)

print()

display(
    df_pilot_ncbi_mapping[
        [
            "Genome ID",
            "Genome Name",
            "Assembly Accession",
            "BioSample Accession",
            "SRA Accession"
        ]
    ]
)


# =========================================================
# 6. Save mapping
# =========================================================

NCBI_MAPPING_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "pilot_ncbi_accession_mapping.csv"
)


df_pilot_ncbi_mapping.to_csv(
    NCBI_MAPPING_PATH,
    index=False
)


print()
print(
    "Mapping saved to:"
)

print(
    NCBI_MAPPING_PATH
)


if assembly_available == 6:

    print()
    print(
        "PASS: All six pilot genomes have "
        "public NCBI assembly accessions."
    )

elif assembly_available > 0:

    print()
    print(
        "PARTIAL: Only some pilot genomes have "
        "public NCBI assembly accessions."
    )

else:

    print()
    print(
        "FAIL: No public NCBI assembly accessions "
        "were identified for the pilot genomes."
    )

[1/6] 1284787.3 | Assembly: GCA_000406405.1
[2/6] 1284788.3 | Assembly: GCA_000401195.1
[3/6] 1284789.4 | Assembly: GCA_000417105.1
[4/6] 1284805.4 | Assembly: GCA_000417545.1
[5/6] 1284814.3 | Assembly: GCA_000408985.1
[6/6] 1284818.4 | Assembly: GCA_000417465.1

PILOT GENOME → NCBI ACCESSION MAPPING
Pilot genomes             : 6
Public assembly accessions: 6/6
BioSample accessions      : 6/6



,Genome ID,Genome Name,Assembly Accession,BioSample Accession,SRA Accession
0,1284787.3,Klebsiella pneumoniae UHKPC40,GCA_000406405.1,SAMN02142015,"SRR849602,SRR900178"
1,1284788.3,Klebsiella pneumoniae UHKPC23,GCA_000401195.1,SAMN02141978,"SRR849539,SRR900161"
2,1284789.4,Klebsiella pneumoniae UHKPC28,GCA_000417105.1,SAMN02141969,"SRR849530,SRR900165"
3,1284805.4,Klebsiella pneumoniae UHKPC17,GCA_000417545.1,SAMN02141975,"SRR849536,SRR900156"
4,1284814.3,Klebsiella pneumoniae UHKPC 52,GCA_000408985.1,SAMN02142019,"SRR849682,SRR863643,SRR921382"
5,1284818.4,Klebsiella pneumoniae UHKPC179,GCA_000417465.1,SAMN02141974,"SRR849535,SRR900157"



Mapping saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/pilot_ncbi_accession_mapping.csv

PASS: All six pilot genomes have public NCBI assembly accessions.


# Cell 7 — Retrieve Precomputed NCBI MicroBIGG-E AMRFinderPlus Results

In [ ]:
import os
import time
import requests
import pandas as pd


# =========================================================
# 1. NCBI MicroBIGG-E query endpoint
# =========================================================

MICROBIGGE_URL = (
    "https://www.ncbi.nlm.nih.gov/pathogens/microbigge/"
)

ncbi_session = requests.Session()

ncbi_session.headers.update(
    {
        "User-Agent": (
            "AMR-Genome-ML/1.0 "
            "(academic research)"
        )
    }
)


# =========================================================
# 2. Pilot assembly accessions
# =========================================================

pilot_assemblies = (
    df_pilot_ncbi_mapping[
        [
            "Genome ID",
            "Assembly Accession"
        ]
    ]
    .dropna()
    .copy()
)


assert len(pilot_assemblies) == 6


# =========================================================
# 3. Query MicroBIGG-E CSV export for each assembly
# =========================================================

microbigge_records = []

microbigge_audit = []


for i, row in pilot_assemblies.iterrows():

    genome_id = str(
        row["Genome ID"]
    )

    assembly = str(
        row["Assembly Accession"]
    )


    # NCBI table export query
    params = {
        "q": f"asm_acc:{assembly}",
        "format": "csv"
    }


    start_time = time.time()


    try:

        response = ncbi_session.get(
            MICROBIGGE_URL,
            params=params,
            timeout=90
        )

        response.raise_for_status()


        content_type = (
            response.headers.get(
                "Content-Type",
                ""
            )
        )


        elapsed = (
            time.time()
            - start_time
        )


        # -------------------------------------------------
        # Save raw response for inspection
        # -------------------------------------------------

        raw_path = os.path.join(
            KNOWN_AMR_DIR,
            f"microbigge_{assembly}.csv"
        )


        with open(
            raw_path,
            "wb"
        ) as f:

            f.write(
                response.content
            )


        # -------------------------------------------------
        # Try reading as CSV
        # -------------------------------------------------

        try:

            df_result = pd.read_csv(
                raw_path
            )

        except Exception:

            df_result = pd.DataFrame()


        # -------------------------------------------------
        # Attach linkage columns
        # -------------------------------------------------

        if not df_result.empty:

            df_result[
                "Genome ID"
            ] = genome_id

            df_result[
                "Assembly Accession"
            ] = assembly


            microbigge_records.append(
                df_result
            )


        microbigge_audit.append(
            {
                "Genome ID":
                    genome_id,

                "Assembly Accession":
                    assembly,

                "Status":
                    "PASS",

                "Rows Retrieved":
                    len(
                        df_result
                    ),

                "HTTP Content Type":
                    content_type,

                "Runtime Seconds":
                    round(
                        elapsed,
                        2
                    ),

                "Raw File":
                    raw_path,

                "Error":
                    None
            }
        )


        print(
            f"[{i + 1}/6] "
            f"{assembly} | "
            f"{len(df_result):,} rows | "
            f"{elapsed:.2f} s"
        )


    except Exception as exc:

        microbigge_audit.append(
            {
                "Genome ID":
                    genome_id,

                "Assembly Accession":
                    assembly,

                "Status":
                    "FAIL",

                "Rows Retrieved":
                    0,

                "HTTP Content Type":
                    None,

                "Runtime Seconds":
                    None,

                "Raw File":
                    None,

                "Error":
                    str(exc)
            }
        )


        print(
            f"[{i + 1}/6] "
            f"{assembly} | FAIL"
        )


# =========================================================
# 4. Combine results
# =========================================================

if microbigge_records:

    df_microbigge_pilot = pd.concat(
        microbigge_records,
        ignore_index=True
    )

else:

    df_microbigge_pilot = pd.DataFrame()


df_microbigge_audit = pd.DataFrame(
    microbigge_audit
)


# =========================================================
# 5. Summary
# =========================================================

print()
print("=" * 70)
print("NCBI MICROBIGG-E PILOT RETRIEVAL")
print("=" * 70)

print(
    f"Pilot assemblies requested : "
    f"{len(pilot_assemblies)}"
)

print(
    f"Successful HTTP requests   : "
    f"{df_microbigge_audit['Status'].eq('PASS').sum()}/6"
)

print(
    f"Total rows retrieved       : "
    f"{len(df_microbigge_pilot):,}"
)


if not df_microbigge_pilot.empty:

    print()

    print(
        "Returned columns:"
    )

    print(
        df_microbigge_pilot
        .columns
        .tolist()
    )


# =========================================================
# 6. Audit table
# =========================================================

print()
display(
    df_microbigge_audit
)


# =========================================================
# 7. Preview results
# =========================================================

if not df_microbigge_pilot.empty:

    print()
    print(
        "Example MicroBIGG-E records:"
    )

    display(
        df_microbigge_pilot
        .head(30)
    )


# =========================================================
# 8. Save combined pilot results
# =========================================================

MICROBIGGE_PILOT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "ncbi_microbigge_pilot_raw.csv"
)

MICROBIGGE_AUDIT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "ncbi_microbigge_pilot_audit.csv"
)


df_microbigge_pilot.to_csv(
    MICROBIGGE_PILOT_PATH,
    index=False
)


df_microbigge_audit.to_csv(
    MICROBIGGE_AUDIT_PATH,
    index=False
)


print()
print(
    "Pilot MicroBIGG-E data saved to:"
)

print(
    MICROBIGGE_PILOT_PATH
)

[1/6] GCA_000406405.1 | 0 rows | 0.26 s
[2/6] GCA_000401195.1 | 0 rows | 0.13 s
[3/6] GCA_000417105.1 | 0 rows | 0.11 s
[4/6] GCA_000417545.1 | 0 rows | 0.11 s
[5/6] GCA_000408985.1 | 0 rows | 0.11 s
[6/6] GCA_000417465.1 | 0 rows | 0.11 s

NCBI MICROBIGG-E PILOT RETRIEVAL
Pilot assemblies requested : 6
Successful HTTP requests   : 6/6
Total rows retrieved       : 0



,Genome ID,Assembly Accession,Status,Rows Retrieved,HTTP Content Type,Runtime Seconds,Raw File,Error
0,1284787.3,GCA_000406405.1,PASS,0,text/html; charset=utf-8,0.26,/content/drive/MyDrive/AMR-Genome-ML/data/feat...,None
1,1284788.3,GCA_000401195.1,PASS,0,text/html; charset=utf-8,0.13,/content/drive/MyDrive/AMR-Genome-ML/data/feat...,None
2,1284789.4,GCA_000417105.1,PASS,0,text/html; charset=utf-8,0.11,/content/drive/MyDrive/AMR-Genome-ML/data/feat...,None
3,1284805.4,GCA_000417545.1,PASS,0,text/html; charset=utf-8,0.11,/content/drive/MyDrive/AMR-Genome-ML/data/feat...,None
4,1284814.3,GCA_000408985.1,PASS,0,text/html; charset=utf-8,0.11,/content/drive/MyDrive/AMR-Genome-ML/data/feat...,None
5,1284818.4,GCA_000417465.1,PASS,0,text/html; charset=utf-8,0.11,/content/drive/MyDrive/AMR-Genome-ML/data/feat...,None



Pilot MicroBIGG-E data saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/ncbi_microbigge_pilot_raw.csv


# Cell 8 — Stream NCBI Klebsiella AMR Results for the Pilot Genomes

In [ ]:
import os
import io
import csv
import requests
import pandas as pd


# =========================================================
# 1. Pin the NCBI Pathogen Detection release
# =========================================================

NCBI_KLEBSIELLA_RELEASE = (
    "PDG000000012.2523"
)

NCBI_AMR_TSV_URL = (
    "https://ftp.ncbi.nlm.nih.gov/pathogen/"
    "Results/Klebsiella/"
    f"{NCBI_KLEBSIELLA_RELEASE}/AMR/"
    f"{NCBI_KLEBSIELLA_RELEASE}.amr.metadata.tsv"
)


# =========================================================
# 2. Pilot identifiers
# =========================================================

pilot_identifier_table = (
    df_pilot_ncbi_mapping[
        [
            "Genome ID",
            "Assembly Accession",
            "BioSample Accession"
        ]
    ]
    .copy()
)


assert len(pilot_identifier_table) == 6


# Search using both assembly and BioSample identifiers
identifier_to_genome = {}


for _, row in pilot_identifier_table.iterrows():

    genome_id = str(
        row["Genome ID"]
    )

    for col in [
        "Assembly Accession",
        "BioSample Accession"
    ]:

        value = row[col]

        if (
            pd.notna(value)
            and str(value).strip()
        ):

            identifier_to_genome[
                str(value).strip()
            ] = genome_id


search_tokens = list(
    identifier_to_genome.keys()
)


print("Identifiers to search:")

for token in search_tokens:
    print(" -", token)


# =========================================================
# 3. Stream the NCBI AMR TSV
# =========================================================

print()
print(
    "Streaming NCBI Klebsiella AMR metadata..."
)


response = requests.get(
    NCBI_AMR_TSV_URL,
    stream=True,
    timeout=120
)

response.raise_for_status()


# =========================================================
# 4. Read header and retain matching lines only
# =========================================================

line_iterator = response.iter_lines(
    decode_unicode=True
)


header_line = next(
    line_iterator
)


header = header_line.split(
    "\t"
)


print()
print(
    f"NCBI columns detected: "
    f"{len(header)}"
)

print(
    header
)


matched_rows = []

matched_tokens = set()

total_lines_scanned = 0


for line in line_iterator:

    if not line:
        continue


    total_lines_scanned += 1


    matching_token = None


    for token in search_tokens:

        if token in line:

            matching_token = token
            break


    if matching_token is None:
        continue


    values = line.split(
        "\t"
    )


    # Pad short rows if necessary
    if len(values) < len(header):

        values += [
            None
        ] * (
            len(header)
            - len(values)
        )


    # Ignore unexpected extra columns safely
    values = values[
        :len(header)
    ]


    record = dict(
        zip(
            header,
            values
        )
    )


    record[
        "Matched Identifier"
    ] = matching_token


    record[
        "Genome ID"
    ] = identifier_to_genome[
        matching_token
    ]


    matched_rows.append(
        record
    )


    matched_tokens.add(
        matching_token
    )


response.close()


# =========================================================
# 5. Construct pilot NCBI AMR table
# =========================================================

df_ncbi_amr_pilot = pd.DataFrame(
    matched_rows
)


# =========================================================
# 6. Retrieval summary
# =========================================================

print()
print("=" * 70)
print("NCBI PATHOGEN DETECTION AMR PILOT RETRIEVAL")
print("=" * 70)

print(
    f"Release                  : "
    f"{NCBI_KLEBSIELLA_RELEASE}"
)

print(
    f"Lines scanned            : "
    f"{total_lines_scanned:,}"
)

print(
    f"Matched AMR rows         : "
    f"{len(df_ncbi_amr_pilot):,}"
)

print(
    f"Pilot genomes represented: "
    f"{(
        df_ncbi_amr_pilot['Genome ID'].nunique()
        if not df_ncbi_amr_pilot.empty
        else 0
    )}/6"
)


# =========================================================
# 7. Counts by pilot genome
# =========================================================

if not df_ncbi_amr_pilot.empty:

    print()
    print(
        "Rows per pilot genome:"
    )

    print(
        df_ncbi_amr_pilot[
            "Genome ID"
        ]
        .value_counts()
        .sort_index()
    )


# =========================================================
# 8. Preview
# =========================================================

if not df_ncbi_amr_pilot.empty:

    print()
    print(
        "Example NCBI AMR records:"
    )

    display(
        df_ncbi_amr_pilot.head(30)
    )


# =========================================================
# 9. Save only matched pilot records
# =========================================================

NCBI_AMR_PILOT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "ncbi_pathogen_detection_amr_pilot.csv"
)


df_ncbi_amr_pilot.to_csv(
    NCBI_AMR_PILOT_PATH,
    index=False
)


print()
print(
    "Pilot NCBI AMR records saved to:"
)

print(
    NCBI_AMR_PILOT_PATH
)


# =========================================================
# 10. Final status
# =========================================================

if (
    not df_ncbi_amr_pilot.empty
    and df_ncbi_amr_pilot[
        "Genome ID"
    ].nunique() == 6
):

    print()
    print(
        "PASS: NCBI AMR records were found "
        "for all six pilot genomes."
    )

elif not df_ncbi_amr_pilot.empty:

    print()
    print(
        "PARTIAL: NCBI AMR records were found "
        "for only some pilot genomes."
    )

else:

    print()
    print(
        "NO MATCHES: The current NCBI AMR result "
        "table did not contain the pilot identifiers."
    )

Identifiers to search:
 - GCA_000406405.1
 - SAMN02142015
 - GCA_000401195.1
 - SAMN02141978
 - GCA_000417105.1
 - SAMN02141969
 - GCA_000417545.1
 - SAMN02141975
 - GCA_000408985.1
 - SAMN02142019
 - GCA_000417465.1
 - SAMN02141974

Streaming NCBI Klebsiella AMR metadata...

NCBI columns detected: 67
['#label', 'FDA_lab_id', 'HHS_region', 'IFSAC_category', 'LibraryLayout', 'PFGE_PrimaryEnzyme_pattern', 'PFGE_SecondaryEnzyme_pattern', 'Platform', 'Run', 'asm_acc', 'asm_level', 'asm_stats_contig_n50', 'asm_stats_length_bp', 'asm_stats_n_contig', 'assembly_method', 'attribute_package', 'bioproject_acc', 'bioproject_center', 'biosample_acc', 'isolate_identifiers', 'collected_by', 'collection_date', 'epi_type', 'fullasm_id', 'geo_loc_name', 'host', 'host_disease', 'isolation_source', 'lat_lon', 'ontological_term', 'outbreak', 'sample_name', 'scientific_name', 'serovar', 'source_type', 'species_taxid', 'sra_center', 'sra_release_date', 'strain', 'sequenced_by', 'project_name', 'food_origin'

,#label,FDA_lab_id,HHS_region,IFSAC_category,LibraryLayout,PFGE_PrimaryEnzyme_pattern,PFGE_SecondaryEnzyme_pattern,Platform,Run,asm_acc,...,number_stress_genes,stress_genotypes,number_virulence_genes,virulence_genotypes,amrfinder_version,refgene_db_version,amrfinder_analysis_type,amrfinder_applied,Matched Identifier,Genome ID
0,PDT000015887.2|SAMN02141969|GCA_000417105.1|Kl...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,GCA_000417105.1,...,7,"""arsC=HMM,asr=HMM,fieF,merA=HMM,merC,merD=HMM,...",0,NULL,4.2.7,2026-01-21.1,COMBINED,1,GCA_000417105.1,1284789.4
1,PDT000015890.2|SAMN02141978|GCA_000401195.1|Kl...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,GCA_000401195.1,...,10,"""arsC=HMM,asr=HMM,clpK,fieF,hsp20,merA=HMM,mer...",0,NULL,4.2.7,2026-01-21.1,COMBINED,1,GCA_000401195.1,1284788.3
2,PDT000015902.2|SAMN02141975|GCA_000417545.1|Kl...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,GCA_000417545.1,...,27,"""arsA,arsB,arsC,arsC=HMM,arsD,arsR,asr=HMM,clp...",0,NULL,4.2.7,2026-01-21.1,COMBINED,1,GCA_000417545.1,1284805.4
3,PDT000015908.2|SAMN02141974|GCA_000417465.1|Kl...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,GCA_000417465.1,...,32,"""arsA,arsB,arsC,arsC=HMM,arsD,arsR,asr=HMM,fie...",0,NULL,4.2.7,2026-01-21.1,COMBINED,1,GCA_000417465.1,1284818.4
4,PDT000016137.2|SAMN02142015|GCA_000406405.1|Kl...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,GCA_000406405.1,...,6,"""arsC=HMM,asr=HMM,clpK,fieF,hsp20,qacL""",0,NULL,4.2.7,2026-01-21.1,COMBINED,1,GCA_000406405.1,1284787.3
5,PDT000016144.2|SAMN02142019|GCA_000408985.1|Kl...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,GCA_000408985.1,...,6,"""arsC=HMM,asr=HMM,clpK,fieF,hsp20,qacEdelta1""",0,NULL,4.2.7,2026-01-21.1,COMBINED,1,GCA_000408985.1,1284814.3
6,PDT000033772.1|SAMN02142019|GCA_000709225.1|Kl...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,GCA_000709225.1,...,6,"""arsC=HMM,asr=HMM,clpK,fieF,hsp20,qacEdelta1""",0,NULL,4.2.7,2026-01-21.1,COMBINED,1,SAMN02142019,1284814.3



Pilot NCBI AMR records saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/ncbi_pathogen_detection_amr_pilot.csv

PASS: NCBI AMR records were found for all six pilot genomes.


# Cell 9 — Compare NCBI Precomputed AMRFinderPlus Genotypes with Local Pilot Results

In [ ]:
import os
import re
import pandas as pd


# =========================================================
# 1. Keep the exact assembly corresponding to each pilot
# =========================================================

df_expected_assemblies = (
    df_pilot_ncbi_mapping[
        [
            "Genome ID",
            "Assembly Accession"
        ]
    ]
    .copy()
)


df_expected_assemblies[
    "Genome ID"
] = (
    df_expected_assemblies[
        "Genome ID"
    ]
    .astype(str)
)


df_ncbi_exact = (
    df_ncbi_amr_pilot
    .merge(
        df_expected_assemblies,
        on="Genome ID",
        how="inner",
        suffixes=(
            "",
            "_expected"
        )
    )
)


df_ncbi_exact = (
    df_ncbi_exact[
        df_ncbi_exact[
            "asm_acc"
        ]
        .astype(str)
        .eq(
            df_ncbi_exact[
                "Assembly Accession"
            ]
            .astype(str)
        )
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(df_ncbi_exact) == 6, (
    f"Expected exactly 6 assembly-matched NCBI rows, "
    f"found {len(df_ncbi_exact)}."
)


assert (
    df_ncbi_exact[
        "Genome ID"
    ]
    .nunique()
    == 6
)


# =========================================================
# 2. Inspect NCBI AMRFinder versions
# =========================================================

print("=" * 70)
print("NCBI PRECOMPUTED AMRFINDERPLUS CONFIGURATION")
print("=" * 70)

print("AMRFinder versions:")

print(
    df_ncbi_exact[
        "amrfinder_version"
    ]
    .value_counts(
        dropna=False
    )
)

print()

print("Reference database versions:")

print(
    df_ncbi_exact[
        "refgene_db_version"
    ]
    .value_counts(
        dropna=False
    )
)

print()

print("Analysis types:")

print(
    df_ncbi_exact[
        "amrfinder_analysis_type"
    ]
    .value_counts(
        dropna=False
    )
)


# =========================================================
# 3. Parse NCBI AMR genotype strings
# =========================================================

def clean_ncbi_genotype(token):

    if pd.isna(token):
        return None

    token = str(token).strip()

    token = token.strip(
        "\"'"
    )

    if not token:
        return None


    # Remove AMRFinder reporting suffixes such as:
    # =POINT, =HMM, =PARTIAL, etc.
    token = re.sub(
        r"=[A-Za-z0-9_+-]+$",
        "",
        token
    )


    token = token.strip()

    return (
        token
        if token
        else None
    )


ncbi_genotype_records = []


for _, row in df_ncbi_exact.iterrows():

    genome_id = str(
        row["Genome ID"]
    )

    genotype_string = row.get(
        "AMR_genotypes"
    )


    if (
        pd.isna(genotype_string)
        or str(genotype_string).strip()
        in {
            "",
            "NULL",
            "None"
        }
    ):

        continue


    tokens = str(
        genotype_string
    ).strip().strip(
        "\""
    ).split(
        ","
    )


    for token in tokens:

        determinant = clean_ncbi_genotype(
            token
        )


        if determinant is None:
            continue


        ncbi_genotype_records.append(
            {
                "Genome ID":
                    genome_id,

                "NCBI Determinant":
                    determinant
            }
        )


df_ncbi_genotypes = (
    pd.DataFrame(
        ncbi_genotype_records
    )
    .drop_duplicates()
    .reset_index(drop=True)
)


# =========================================================
# 4. Prepare local AMRFinder reference
# =========================================================

df_local_reference = (
    df_amrfinder_pilot_hits[
        df_amrfinder_pilot_hits[
            "Type"
        ]
        .astype(str)
        .str.upper()
        .eq("AMR")
    ][
        [
            "Genome ID",
            "Element symbol",
            "Subtype",
            "Class"
        ]
    ]
    .copy()
)


df_local_reference[
    "Genome ID"
] = (
    df_local_reference[
        "Genome ID"
    ]
    .astype(str)
)


assert len(df_local_reference) == 99


# =========================================================
# 5. Normalization for exact comparison
# =========================================================

def normalize_determinant(value):

    if pd.isna(value):
        return None

    return re.sub(
        r"\s+",
        "",
        str(value).strip().lower()
    )


df_local_reference[
    "Normalized"
] = (
    df_local_reference[
        "Element symbol"
    ]
    .apply(
        normalize_determinant
    )
)


df_ncbi_genotypes[
    "Normalized"
] = (
    df_ncbi_genotypes[
        "NCBI Determinant"
    ]
    .apply(
        normalize_determinant
    )
)


# =========================================================
# 6. Compare occurrence-by-occurrence
# =========================================================

ncbi_pairs = set(
    zip(
        df_ncbi_genotypes[
            "Genome ID"
        ],
        df_ncbi_genotypes[
            "Normalized"
        ]
    )
)


df_local_reference[
    "Found in NCBI"
] = [
    (
        genome_id,
        determinant
    )
    in ncbi_pairs

    for genome_id, determinant
    in zip(
        df_local_reference[
            "Genome ID"
        ],
        df_local_reference[
            "Normalized"
        ]
    )
]


matched_local = int(
    df_local_reference[
        "Found in NCBI"
    ]
    .sum()
)


total_local = len(
    df_local_reference
)


local_coverage = (
    matched_local
    / total_local
)


# =========================================================
# 7. Find NCBI calls absent from local pilot
# =========================================================

local_pairs = set(
    zip(
        df_local_reference[
            "Genome ID"
        ],
        df_local_reference[
            "Normalized"
        ]
    )
)


df_ncbi_genotypes[
    "Found Locally"
] = [
    (
        genome_id,
        determinant
    )
    in local_pairs

    for genome_id, determinant
    in zip(
        df_ncbi_genotypes[
            "Genome ID"
        ],
        df_ncbi_genotypes[
            "Normalized"
        ]
    )
]


ncbi_extra = (
    df_ncbi_genotypes[
        ~df_ncbi_genotypes[
            "Found Locally"
        ]
    ]
    .copy()
)


# =========================================================
# 8. Mutation-specific comparison
# =========================================================

mutation_mask = (
    df_local_reference[
        "Subtype"
    ]
    .isin(
        [
            "POINT",
            "POINT_DISRUPT"
        ]
    )
)


df_local_mutations = (
    df_local_reference[
        mutation_mask
    ]
)


mutation_total = len(
    df_local_mutations
)


mutation_matched = int(
    df_local_mutations[
        "Found in NCBI"
    ]
    .sum()
)


# =========================================================
# 9. Per-genome summary
# =========================================================

local_counts = (
    df_local_reference
    .groupby(
        "Genome ID"
    )
    .size()
    .rename(
        "Local AMR Calls"
    )
)


ncbi_counts = (
    df_ncbi_genotypes
    .groupby(
        "Genome ID"
    )
    .size()
    .rename(
        "NCBI AMR Calls"
    )
)


matched_counts = (
    df_local_reference[
        df_local_reference[
            "Found in NCBI"
        ]
    ]
    .groupby(
        "Genome ID"
    )
    .size()
    .rename(
        "Shared Calls"
    )
)


df_genome_comparison = (
    pd.concat(
        [
            local_counts,
            ncbi_counts,
            matched_counts
        ],
        axis=1
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)


# =========================================================
# 10. Save comparison
# =========================================================

NCBI_COMPARISON_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "local_vs_ncbi_amrfinder_pilot_comparison.csv"
)


df_local_reference.to_csv(
    NCBI_COMPARISON_PATH,
    index=False
)


# =========================================================
# 11. Summary
# =========================================================

print()
print("=" * 70)
print("LOCAL vs NCBI AMRFINDERPLUS PILOT COMPARISON")
print("=" * 70)

print(
    f"Local AMR occurrences      : "
    f"{total_local}"
)

print(
    f"NCBI AMR occurrences       : "
    f"{len(df_ncbi_genotypes)}"
)

print(
    f"Local calls found in NCBI  : "
    f"{matched_local}/{total_local} "
    f"({local_coverage:.1%})"
)

print()

print(
    f"Local mutation occurrences : "
    f"{mutation_total}"
)

print(
    f"Mutations found in NCBI    : "
    f"{mutation_matched}/{mutation_total}"
)

print()

print("Per-genome comparison:")

display(
    df_genome_comparison
)


# =========================================================
# 12. Local calls missing from NCBI
# =========================================================

df_local_missing_ncbi = (
    df_local_reference[
        ~df_local_reference[
            "Found in NCBI"
        ]
    ]
    .copy()
)


print()
print(
    f"Local AMRFinder calls missing from NCBI: "
    f"{len(df_local_missing_ncbi)}"
)

display(
    df_local_missing_ncbi[
        [
            "Genome ID",
            "Element symbol",
            "Subtype",
            "Class"
        ]
    ]
)


# =========================================================
# 13. NCBI-only calls
# =========================================================

print()
print(
    f"NCBI calls absent from local pilot: "
    f"{len(ncbi_extra)}"
)


if not ncbi_extra.empty:

    display(
        ncbi_extra[
            [
                "Genome ID",
                "NCBI Determinant"
            ]
        ]
    )


print()
print(
    "Comparison saved to:"
)

print(
    NCBI_COMPARISON_PATH
)

print()

print(
    "PASS: Exact pilot comparison between local and "
    "NCBI precomputed AMRFinderPlus results is complete."
)

NCBI PRECOMPUTED AMRFINDERPLUS CONFIGURATION
AMRFinder versions:
amrfinder_version
4.2.7    6
Name: count, dtype: int64

Reference database versions:
refgene_db_version
2026-01-21.1    6
Name: count, dtype: int64

Analysis types:
amrfinder_analysis_type
COMBINED    6
Name: count, dtype: int64

LOCAL vs NCBI AMRFINDERPLUS PILOT COMPARISON
Local AMR occurrences      : 99
NCBI AMR occurrences       : 105
Local calls found in NCBI  : 99/99 (100.0%)

Local mutation occurrences : 26
Mutations found in NCBI    : 26/26

Per-genome comparison:


,Genome ID,Local AMR Calls,NCBI AMR Calls,Shared Calls
0,1284787.3,16,17,16
1,1284788.3,17,18,17
2,1284789.4,12,13,12
3,1284805.4,22,23,22
4,1284814.3,14,15,14
5,1284818.4,18,19,18



Local AMRFinder calls missing from NCBI: 0


,Genome ID,Element symbol,Subtype,Class



NCBI calls absent from local pilot: 6


,Genome ID,NCBI Determinant
5,1284789.4,emrD
20,1284788.3,emrD
41,1284805.4,emrD
62,1284818.4,emrD
80,1284787.3,emrD
94,1284814.3,emrD



Comparison saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/local_vs_ncbi_amrfinder_pilot_comparison.csv

PASS: Exact pilot comparison between local and NCBI precomputed AMRFinderPlus results is complete.


# Cell 10 — Map the Full Cohort to NCBI Assembly Accessions

In [ ]:
import os
import time
import requests
import pandas as pd


# =========================================================
# 1. BV-BRC genome API
# =========================================================

BV_BRC_GENOME_API = (
    "https://www.bv-brc.org/api/genome/"
)

session = requests.Session()

session.headers.update(
    {
        "Accept": "application/json",
        "User-Agent": (
            "AMR-Genome-ML/1.0 "
            "(academic research)"
        )
    }
)


# =========================================================
# 2. Full QC-qualified cohort
# =========================================================

full_genome_ids = (
    df_feature_input[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)


assert len(full_genome_ids) == 4233


# =========================================================
# 3. Batch configuration
# =========================================================

BATCH_SIZE = 200

mapping_records = []
batch_audit = []


batches = [
    full_genome_ids[i:i + BATCH_SIZE]
    for i in range(
        0,
        len(full_genome_ids),
        BATCH_SIZE
    )
]


print(
    f"Total genomes : {len(full_genome_ids):,}"
)

print(
    f"Batch size    : {BATCH_SIZE}"
)

print(
    f"Total batches : {len(batches)}"
)

print()


# =========================================================
# 4. Retrieve mappings in batches
# =========================================================

for batch_index, batch_ids in enumerate(
    batches,
    start=1
):

    id_string = ",".join(
        batch_ids
    )


    query_url = (
        f"{BV_BRC_GENOME_API}"
        f"?in(genome_id,({id_string}))"
        f"&select("
        f"genome_id,"
        f"genome_name,"
        f"assembly_accession,"
        f"biosample_accession"
        f")"
        f"&limit(10000)"
    )


    success = False
    last_error = None
    start_time = time.time()


    # ---------------------------------------------
    # Retry transient API failures
    # ---------------------------------------------

    for attempt in range(
        1,
        4
    ):

        try:

            response = session.get(
                query_url,
                timeout=120
            )

            response.raise_for_status()

            records = response.json()


            if not isinstance(
                records,
                list
            ):

                raise RuntimeError(
                    "Unexpected BV-BRC response format."
                )


            mapping_records.extend(
                records
            )

            success = True

            break


        except Exception as exc:

            last_error = str(
                exc
            )

            if attempt < 3:

                time.sleep(
                    2 * attempt
                )


    elapsed = (
        time.time()
        - start_time
    )


    batch_audit.append(
        {
            "Batch":
                batch_index,

            "Genome IDs Requested":
                len(batch_ids),

            "Records Returned":
                (
                    len(records)
                    if success
                    else 0
                ),

            "Status":
                (
                    "PASS"
                    if success
                    else "FAIL"
                ),

            "Runtime Seconds":
                round(
                    elapsed,
                    2
                ),

            "Error":
                (
                    None
                    if success
                    else last_error
                )
        }
    )


    print(
        f"[{batch_index:02d}/{len(batches)}] "
        f"requested={len(batch_ids):3d} | "
        f"returned="
        f"{len(records) if success else 0:3d} | "
        f"{'PASS' if success else 'FAIL'} | "
        f"{elapsed:.2f} s"
    )


# =========================================================
# 5. Verify batch retrieval
# =========================================================

df_mapping_batch_audit = pd.DataFrame(
    batch_audit
)


failed_batches = (
    df_mapping_batch_audit[
        df_mapping_batch_audit[
            "Status"
        ]
        .ne("PASS")
    ]
)


if not failed_batches.empty:

    display(
        failed_batches
    )

    raise RuntimeError(
        "One or more BV-BRC mapping batches failed."
    )


# =========================================================
# 6. Construct returned metadata table
# =========================================================

df_bvbrc_ncbi_metadata = pd.DataFrame(
    mapping_records
)


if not df_bvbrc_ncbi_metadata.empty:

    df_bvbrc_ncbi_metadata[
        "genome_id"
    ] = (
        df_bvbrc_ncbi_metadata[
            "genome_id"
        ]
        .astype(str)
        .str.strip()
    )


    df_bvbrc_ncbi_metadata = (
        df_bvbrc_ncbi_metadata
        .drop_duplicates(
            subset=[
                "genome_id"
            ]
        )
        .reset_index(
            drop=True
        )
    )


# =========================================================
# 7. Merge against the exact study cohort
# =========================================================

df_full_ncbi_mapping = (
    df_feature_input[
        [
            "Genome ID",
            "Genome Name"
        ]
    ]
    .copy()
)


df_full_ncbi_mapping[
    "Genome ID"
] = (
    df_full_ncbi_mapping[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


df_full_ncbi_mapping = (
    df_full_ncbi_mapping
    .merge(
        df_bvbrc_ncbi_metadata[
            [
                "genome_id",
                "assembly_accession",
                "biosample_accession"
            ]
        ],
        left_on="Genome ID",
        right_on="genome_id",
        how="left"
    )
    .drop(
        columns=[
            "genome_id"
        ]
    )
    .rename(
        columns={
            "assembly_accession":
                "Assembly Accession",

            "biosample_accession":
                "BioSample Accession"
        }
    )
)


assert len(
    df_full_ncbi_mapping
) == 4233


assert (
    df_full_ncbi_mapping[
        "Genome ID"
    ]
    .nunique()
    == 4233
)


# =========================================================
# 8. Clean empty accession values
# =========================================================

for col in [
    "Assembly Accession",
    "BioSample Accession"
]:

    df_full_ncbi_mapping[
        col
    ] = (
        df_full_ncbi_mapping[
            col
        ]
        .replace(
            {
                "":
                    pd.NA,

                "NULL":
                    pd.NA,

                "None":
                    pd.NA,

                "nan":
                    pd.NA
            }
        )
    )


# =========================================================
# 9. Coverage statistics
# =========================================================

assembly_count = int(
    df_full_ncbi_mapping[
        "Assembly Accession"
    ]
    .notna()
    .sum()
)


biosample_count = int(
    df_full_ncbi_mapping[
        "BioSample Accession"
    ]
    .notna()
    .sum()
)


assembly_rate = (
    assembly_count
    / len(
        df_full_ncbi_mapping
    )
)


biosample_rate = (
    biosample_count
    / len(
        df_full_ncbi_mapping
    )
)


# =========================================================
# 10. Save persistent mapping
# =========================================================

FULL_NCBI_MAPPING_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "full_cohort_ncbi_accession_mapping.csv"
)


FULL_NCBI_MAPPING_AUDIT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "full_cohort_ncbi_mapping_batch_audit.csv"
)


df_full_ncbi_mapping.to_csv(
    FULL_NCBI_MAPPING_PATH,
    index=False
)


df_mapping_batch_audit.to_csv(
    FULL_NCBI_MAPPING_AUDIT_PATH,
    index=False
)


# =========================================================
# 11. Final summary
# =========================================================

print()
print("=" * 70)
print("FULL-COHORT NCBI ACCESSION MAPPING")
print("=" * 70)

print(
    f"Study genomes             : "
    f"{len(df_full_ncbi_mapping):,}"
)

print(
    f"BV-BRC records retrieved  : "
    f"{df_bvbrc_ncbi_metadata['genome_id'].nunique():,}"
)

print(
    f"Assembly accessions       : "
    f"{assembly_count:,}/4,233 "
    f"({assembly_rate:.2%})"
)

print(
    f"BioSample accessions      : "
    f"{biosample_count:,}/4,233 "
    f"({biosample_rate:.2%})"
)


missing_assembly = (
    df_full_ncbi_mapping[
        df_full_ncbi_mapping[
            "Assembly Accession"
        ]
        .isna()
    ]
)


print(
    f"Missing assembly accession: "
    f"{len(missing_assembly):,}"
)


if not missing_assembly.empty:

    print()
    print(
        "Example genomes without an assembly accession:"
    )

    display(
        missing_assembly.head(20)
    )


print()
print(
    "Mapping saved to:"
)

print(
    FULL_NCBI_MAPPING_PATH
)

print()

print(
    "PASS: Full-cohort NCBI accession mapping "
    "is complete."
)

Total genomes : 4,233
Batch size    : 200
Total batches : 22

[01/22] requested=200 | returned=200 | PASS | 0.41 s
[02/22] requested=200 | returned=200 | PASS | 0.20 s
[03/22] requested=200 | returned=200 | PASS | 0.19 s
[04/22] requested=200 | returned=200 | PASS | 0.21 s
[05/22] requested=200 | returned=200 | PASS | 0.20 s
[06/22] requested=200 | returned=200 | PASS | 0.20 s
[07/22] requested=200 | returned=200 | PASS | 0.20 s
[08/22] requested=200 | returned=200 | PASS | 0.18 s
[09/22] requested=200 | returned=200 | PASS | 0.19 s
[10/22] requested=200 | returned=200 | PASS | 0.18 s
[11/22] requested=200 | returned=200 | PASS | 0.19 s
[12/22] requested=200 | returned=200 | PASS | 0.18 s
[13/22] requested=200 | returned=200 | PASS | 0.19 s
[14/22] requested=200 | returned=200 | PASS | 0.18 s
[15/22] requested=200 | returned=200 | PASS | 0.18 s
[16/22] requested=200 | returned=200 | PASS | 0.19 s
[17/22] requested=200 | returned=200 | PASS | 0.22 s
[18/22] requested=200 | returned=200 

,Genome ID,Genome Name,Assembly Accession,BioSample Accession
149,573.12858,Klebsiella pneumoniae 3243,NaN,SAMN07525473
150,573.12859,Klebsiella pneumoniae 3248,NaN,SAMN07525475
151,573.12860,Klebsiella pneumoniae 3316,NaN,SAMN07525518
152,573.12861,Klebsiella pneumoniae 35,NaN,SAMN07525396
153,573.12862,Klebsiella pneumoniae 3288,NaN,SAMN07525498
154,573.12863,Klebsiella pneumoniae 105,NaN,SAMN07525444
155,573.12864,Klebsiella pneumoniae 3376,NaN,SAMN07525543
156,573.12865,Klebsiella pneumoniae 34,NaN,SAMN07525395
157,573.12867,Klebsiella pneumoniae 3345,NaN,SAMN07525532
158,573.12868,Klebsiella pneumoniae 78,NaN,SAMN07525423



Mapping saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/full_cohort_ncbi_accession_mapping.csv

PASS: Full-cohort NCBI accession mapping is complete.


# Cell 11 — Resolve Full-Cohort NCBI AMR Coverage

In [ ]:
import os
import csv
import requests
import pandas as pd


# =========================================================
# 1. NCBI release
# =========================================================

NCBI_KLEBSIELLA_RELEASE = (
    "PDG000000012.2523"
)

NCBI_AMR_TSV_URL = (
    "https://ftp.ncbi.nlm.nih.gov/pathogen/"
    "Results/Klebsiella/"
    f"{NCBI_KLEBSIELLA_RELEASE}/AMR/"
    f"{NCBI_KLEBSIELLA_RELEASE}.amr.metadata.tsv"
)


# =========================================================
# 2. Prepare study identifiers
# =========================================================

df_mapping = (
    df_full_ncbi_mapping
    .copy()
)


for col in [
    "Genome ID",
    "Assembly Accession",
    "BioSample Accession"
]:

    df_mapping[col] = (
        df_mapping[col]
        .astype("string")
        .str.strip()
    )


study_assemblies = set(
    df_mapping[
        "Assembly Accession"
    ]
    .dropna()
    .tolist()
)


study_biosamples = set(
    df_mapping[
        "BioSample Accession"
    ]
    .dropna()
    .tolist()
)


print(
    f"Study assembly identifiers : "
    f"{len(study_assemblies):,}"
)

print(
    f"Study BioSample identifiers: "
    f"{len(study_biosamples):,}"
)


# =========================================================
# 3. Stream NCBI table once
# =========================================================

print()
print(
    "Streaming NCBI Klebsiella AMR metadata..."
)


response = requests.get(
    NCBI_AMR_TSV_URL,
    stream=True,
    timeout=120
)

response.raise_for_status()


line_iterator = response.iter_lines(
    decode_unicode=True
)


header_line = next(
    line_iterator
)

header = header_line.split(
    "\t"
)


asm_idx = header.index(
    "asm_acc"
)

biosample_idx = header.index(
    "biosample_acc"
)


matched_ncbi_rows = []

total_lines_scanned = 0


for line in line_iterator:

    if not line:
        continue


    total_lines_scanned += 1


    values = line.split(
        "\t"
    )


    if len(values) < len(header):

        values += [
            None
        ] * (
            len(header)
            - len(values)
        )


    values = values[
        :len(header)
    ]


    asm_acc = values[
        asm_idx
    ]

    biosample_acc = values[
        biosample_idx
    ]


    asm_match = (
        asm_acc
        in study_assemblies
    )


    biosample_match = (
        biosample_acc
        in study_biosamples
    )


    if not (
        asm_match
        or biosample_match
    ):

        continue


    matched_ncbi_rows.append(
        dict(
            zip(
                header,
                values
            )
        )
    )


response.close()


df_ncbi_full_candidates = pd.DataFrame(
    matched_ncbi_rows
)


print()
print(
    f"Lines scanned             : "
    f"{total_lines_scanned:,}"
)

print(
    f"Candidate NCBI rows       : "
    f"{len(df_ncbi_full_candidates):,}"
)


# =========================================================
# 4. Clean NCBI identifiers
# =========================================================

for col in [
    "asm_acc",
    "biosample_acc",
    "AMR_genotypes",
    "amrfinder_version",
    "refgene_db_version",
    "amrfinder_analysis_type"
]:

    if col in df_ncbi_full_candidates.columns:

        df_ncbi_full_candidates[col] = (
            df_ncbi_full_candidates[col]
            .astype("string")
            .str.strip()
        )


# =========================================================
# 5. Helper for missing text
# =========================================================

def clean_text(value):

    if pd.isna(value):
        return None

    value = str(
        value
    ).strip()


    if value in {
        "",
        "NULL",
        "None",
        "nan",
        "<NA>"
    }:

        return None


    return value


# =========================================================
# 6. Build lookup dictionaries
# =========================================================

assembly_lookup = {}

biosample_lookup = {}


for idx, row in df_ncbi_full_candidates.iterrows():

    asm = clean_text(
        row.get(
            "asm_acc"
        )
    )

    biosample = clean_text(
        row.get(
            "biosample_acc"
        )
    )


    if asm is not None:

        assembly_lookup.setdefault(
            asm,
            []
        ).append(
            idx
        )


    if biosample is not None:

        biosample_lookup.setdefault(
            biosample,
            []
        ).append(
            idx
        )


# =========================================================
# 7. Resolve each study genome conservatively
# =========================================================

resolution_records = []


for _, row in df_mapping.iterrows():

    genome_id = str(
        row[
            "Genome ID"
        ]
    )

    assembly = clean_text(
        row[
            "Assembly Accession"
        ]
    )

    biosample = clean_text(
        row[
            "BioSample Accession"
        ]
    )


    selected_index = None

    status = None

    candidate_count = 0


    # -----------------------------------------------------
    # Priority 1 — exact assembly accession
    # -----------------------------------------------------

    if (
        assembly is not None
        and assembly in assembly_lookup
    ):

        candidate_indices = (
            assembly_lookup[
                assembly
            ]
        )

        candidate_count = len(
            candidate_indices
        )


        if candidate_count == 1:

            selected_index = (
                candidate_indices[0]
            )

            status = (
                "EXACT_ASSEMBLY"
            )


        else:

            subset = (
                df_ncbi_full_candidates
                .loc[
                    candidate_indices
                ]
            )


            genotype_values = (
                subset[
                    "AMR_genotypes"
                ]
                .fillna("")
                .astype(str)
                .unique()
            )


            if len(
                genotype_values
            ) == 1:

                selected_index = (
                    candidate_indices[0]
                )

                status = (
                    "EXACT_ASSEMBLY_DUPLICATE_IDENTICAL"
                )

            else:

                status = (
                    "EXACT_ASSEMBLY_AMBIGUOUS"
                )


    # -----------------------------------------------------
    # Priority 2 — BioSample
    # -----------------------------------------------------

    elif (
        biosample is not None
        and biosample in biosample_lookup
    ):

        candidate_indices = (
            biosample_lookup[
                biosample
            ]
        )

        candidate_count = len(
            candidate_indices
        )


        if candidate_count == 1:

            selected_index = (
                candidate_indices[0]
            )

            status = (
                "BIOSAMPLE_UNIQUE"
            )


        else:

            subset = (
                df_ncbi_full_candidates
                .loc[
                    candidate_indices
                ]
            )


            # Compare both AMR calls and AMRFinder provenance
            comparison_columns = [
                "AMR_genotypes",
                "amrfinder_version",
                "refgene_db_version",
                "amrfinder_analysis_type"
            ]


            signatures = (
                subset[
                    comparison_columns
                ]
                .fillna("")
                .astype(str)
                .agg(
                    " || ".join,
                    axis=1
                )
                .unique()
            )


            if len(
                signatures
            ) == 1:

                selected_index = (
                    candidate_indices[0]
                )

                status = (
                    "BIOSAMPLE_MULTIPLE_IDENTICAL"
                )

            else:

                status = (
                    "BIOSAMPLE_AMBIGUOUS"
                )


    # -----------------------------------------------------
    # No NCBI match
    # -----------------------------------------------------

    else:

        status = (
            "NO_NCBI_MATCH"
        )


    # -----------------------------------------------------
    # Extract selected NCBI record
    # -----------------------------------------------------

    selected_assembly = None
    selected_biosample = None
    amr_genotypes = None
    amrfinder_version = None
    refgene_db_version = None
    analysis_type = None


    if selected_index is not None:

        selected_row = (
            df_ncbi_full_candidates
            .loc[
                selected_index
            ]
        )


        selected_assembly = clean_text(
            selected_row.get(
                "asm_acc"
            )
        )

        selected_biosample = clean_text(
            selected_row.get(
                "biosample_acc"
            )
        )

        amr_genotypes = clean_text(
            selected_row.get(
                "AMR_genotypes"
            )
        )

        amrfinder_version = clean_text(
            selected_row.get(
                "amrfinder_version"
            )
        )

        refgene_db_version = clean_text(
            selected_row.get(
                "refgene_db_version"
            )
        )

        analysis_type = clean_text(
            selected_row.get(
                "amrfinder_analysis_type"
            )
        )


    resolution_records.append(
        {
            "Genome ID":
                genome_id,

            "Study Assembly Accession":
                assembly,

            "Study BioSample Accession":
                biosample,

            "Resolution Status":
                status,

            "NCBI Candidate Rows":
                candidate_count,

            "Selected NCBI Assembly":
                selected_assembly,

            "Selected NCBI BioSample":
                selected_biosample,

            "AMR Genotypes":
                amr_genotypes,

            "AMRFinder Version":
                amrfinder_version,

            "Reference DB Version":
                refgene_db_version,

            "Analysis Type":
                analysis_type
        }
    )


df_ncbi_resolution = pd.DataFrame(
    resolution_records
)


assert len(
    df_ncbi_resolution
) == 4233


assert (
    df_ncbi_resolution[
        "Genome ID"
    ]
    .nunique()
    == 4233
)


# =========================================================
# 8. Define scientifically usable matches
# =========================================================

USABLE_STATUSES = {
    "EXACT_ASSEMBLY",
    "EXACT_ASSEMBLY_DUPLICATE_IDENTICAL",
    "BIOSAMPLE_UNIQUE",
    "BIOSAMPLE_MULTIPLE_IDENTICAL"
}


df_ncbi_resolution[
    "Usable NCBI AMR"
] = (
    df_ncbi_resolution[
        "Resolution Status"
    ]
    .isin(
        USABLE_STATUSES
    )
)


# =========================================================
# 9. Coverage summary
# =========================================================

usable_count = int(
    df_ncbi_resolution[
        "Usable NCBI AMR"
    ]
    .sum()
)


usable_rate = (
    usable_count
    / len(
        df_ncbi_resolution
    )
)


print()
print("=" * 70)
print("FULL-COHORT NCBI AMR RESOLUTION")
print("=" * 70)

print(
    f"Study genomes              : "
    f"{len(df_ncbi_resolution):,}"
)

print(
    f"Usable NCBI AMR results    : "
    f"{usable_count:,}/4,233 "
    f"({usable_rate:.2%})"
)

print()

print(
    "Resolution status:"
)

print(
    df_ncbi_resolution[
        "Resolution Status"
    ]
    .value_counts(
        dropna=False
    )
)


# =========================================================
# 10. AMRFinder provenance among usable rows
# =========================================================

df_usable_ncbi = (
    df_ncbi_resolution[
        df_ncbi_resolution[
            "Usable NCBI AMR"
        ]
    ]
)


print()
print(
    "AMRFinder versions among usable records:"
)

print(
    df_usable_ncbi[
        "AMRFinder Version"
    ]
    .value_counts(
        dropna=False
    )
)


print()
print(
    "Reference database versions:"
)

print(
    df_usable_ncbi[
        "Reference DB Version"
    ]
    .value_counts(
        dropna=False
    )
)


print()
print(
    "Analysis types:"
)

print(
    df_usable_ncbi[
        "Analysis Type"
    ]
    .value_counts(
        dropna=False
    )
)


# =========================================================
# 11. Inspect unresolved cases
# =========================================================

df_unresolved_ncbi = (
    df_ncbi_resolution[
        ~df_ncbi_resolution[
            "Usable NCBI AMR"
        ]
    ]
    .copy()
)


print()
print(
    f"Unresolved genomes         : "
    f"{len(df_unresolved_ncbi):,}"
)


if not df_unresolved_ncbi.empty:

    display(
        df_unresolved_ncbi[
            [
                "Genome ID",
                "Study Assembly Accession",
                "Study BioSample Accession",
                "Resolution Status",
                "NCBI Candidate Rows"
            ]
        ]
        .head(30)
    )


# =========================================================
# 12. Save persistent outputs
# =========================================================

NCBI_FULL_CANDIDATES_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "ncbi_full_cohort_candidate_rows.csv"
)

NCBI_RESOLUTION_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "ncbi_full_cohort_amr_resolution.csv"
)


df_ncbi_full_candidates.to_csv(
    NCBI_FULL_CANDIDATES_PATH,
    index=False
)


df_ncbi_resolution.to_csv(
    NCBI_RESOLUTION_PATH,
    index=False
)


print()
print(
    "Candidate rows saved to:"
)

print(
    NCBI_FULL_CANDIDATES_PATH
)

print()

print(
    "Resolution table saved to:"
)

print(
    NCBI_RESOLUTION_PATH
)

print()

print(
    "PASS: Full-cohort NCBI AMR matching "
    "and ambiguity audit are complete."
)

Study assembly identifiers : 951
Study BioSample identifiers: 4,151

Streaming NCBI Klebsiella AMR metadata...

Lines scanned             : 174,631
Candidate NCBI rows       : 3,843

FULL-COHORT NCBI AMR RESOLUTION
Study genomes              : 4,233
Usable NCBI AMR results    : 3,851/4,233 (90.98%)

Resolution status:
Resolution Status
BIOSAMPLE_UNIQUE    3169
EXACT_ASSEMBLY       682
NO_NCBI_MATCH        382
Name: count, dtype: int64

AMRFinder versions among usable records:
AMRFinder Version
4.2.7    3851
Name: count, dtype: int64

Reference database versions:
Reference DB Version
2026-01-21.1    3847
2026-05-15.1       4
Name: count, dtype: int64

Analysis types:
Analysis Type
COMBINED      3053
NUCLEOTIDE     798
Name: count, dtype: int64

Unresolved genomes         : 382


,Genome ID,Study Assembly Accession,Study BioSample Accession,Resolution Status,NCBI Candidate Rows
15,1284802.4,GCA_000417085.1,SAMN02142014,NO_NCBI_MATCH,0
174,573.12886,None,SAMN07525407,NO_NCBI_MATCH,0
314,573.13037,None,SAMN06437890,NO_NCBI_MATCH,0
317,573.13040,None,SAMN06437893,NO_NCBI_MATCH,0
318,573.13041,None,SAMN06437894,NO_NCBI_MATCH,0
325,573.13049,None,SAMN06437902,NO_NCBI_MATCH,0
341,573.13065,None,SAMN06437919,NO_NCBI_MATCH,0
348,573.13073,None,SAMN06437929,NO_NCBI_MATCH,0
359,573.13086,None,SAMN06437944,NO_NCBI_MATCH,0
365,573.13092,None,SAMN06437950,NO_NCBI_MATCH,0



Candidate rows saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/ncbi_full_cohort_candidate_rows.csv

Resolution table saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/ncbi_full_cohort_amr_resolution.csv

PASS: Full-cohort NCBI AMR matching and ambiguity audit are complete.


# Cell 12 — Audit Unresolved NCBI Coverage and Local Completion Cost

In [ ]:
import os
import glob
import pandas as pd


# =========================================================
# 1. Attach phenotype for coverage-bias audit only
# =========================================================

df_resolution_audit = (
    df_ncbi_resolution
    .merge(
        df_feature_input[
            [
                "Genome ID",
                "Resistant Phenotype"
            ]
        ],
        on="Genome ID",
        how="left"
    )
)


assert len(
    df_resolution_audit
) == 4233


assert (
    df_resolution_audit[
        "Resistant Phenotype"
    ]
    .notna()
    .all()
)


# =========================================================
# 2. Separate resolved and unresolved genomes
# =========================================================

df_resolved = (
    df_resolution_audit[
        df_resolution_audit[
            "Usable NCBI AMR"
        ]
    ]
    .copy()
)


df_unresolved = (
    df_resolution_audit[
        ~df_resolution_audit[
            "Usable NCBI AMR"
        ]
    ]
    .copy()
)


assert len(
    df_unresolved
) == 382


# =========================================================
# 3. Phenotype distribution
# =========================================================

print("=" * 70)
print("NCBI COVERAGE BIAS AUDIT")
print("=" * 70)

print(
    f"Resolved genomes   : "
    f"{len(df_resolved):,}"
)

print(
    f"Unresolved genomes : "
    f"{len(df_unresolved):,}"
)


print()
print("Resolved phenotype distribution:")

print(
    df_resolved[
        "Resistant Phenotype"
    ]
    .value_counts()
)


print()
print("Unresolved phenotype distribution:")

print(
    df_unresolved[
        "Resistant Phenotype"
    ]
    .value_counts()
)


# =========================================================
# 4. Resolution rates within phenotype groups
# =========================================================

coverage_by_phenotype = (
    df_resolution_audit
    .groupby(
        "Resistant Phenotype"
    )[
        "Usable NCBI AMR"
    ]
    .agg(
        [
            "count",
            "sum"
        ]
    )
    .rename(
        columns={
            "count":
                "Total",

            "sum":
                "Resolved"
        }
    )
)


coverage_by_phenotype[
    "Unresolved"
] = (
    coverage_by_phenotype[
        "Total"
    ]
    - coverage_by_phenotype[
        "Resolved"
    ]
)


coverage_by_phenotype[
    "Resolution Rate"
] = (
    coverage_by_phenotype[
        "Resolved"
    ]
    / coverage_by_phenotype[
        "Total"
    ]
)


print()
print(
    "NCBI resolution rate by phenotype:"
)

display(
    coverage_by_phenotype
)


# =========================================================
# 5. Check existing local AMRFinder outputs
# =========================================================

RAW_AMRFINDER_DIR = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "known_amr",
    "raw_amrfinder"
)


os.makedirs(
    RAW_AMRFINDER_DIR,
    exist_ok=True
)


existing_tsv_files = glob.glob(
    os.path.join(
        RAW_AMRFINDER_DIR,
        "*.amrfinder.tsv"
    )
)


existing_output_ids = {
    os.path.basename(
        path
    ).replace(
        ".amrfinder.tsv",
        ""
    )
    for path in existing_tsv_files
}


unresolved_ids = set(
    df_unresolved[
        "Genome ID"
    ]
    .astype(str)
)


already_completed_unresolved = (
    unresolved_ids
    & existing_output_ids
)


remaining_local_ids = (
    unresolved_ids
    - existing_output_ids
)


# =========================================================
# 6. Runtime estimate
# =========================================================

NUCLEOTIDE_RUNTIME_SECONDS = 35.20


estimated_seconds = (
    len(
        remaining_local_ids
    )
    * NUCLEOTIDE_RUNTIME_SECONDS
)


estimated_hours = (
    estimated_seconds
    / 3600
)


full_cohort_hours = (
    4233
    * NUCLEOTIDE_RUNTIME_SECONDS
    / 3600
)


time_reduction = (
    1
    - (
        estimated_hours
        / full_cohort_hours
    )
)


# =========================================================
# 7. Prepare local-completion manifest
# =========================================================

df_local_completion = (
    df_feature_input[
        df_feature_input[
            "Genome ID"
        ]
        .astype(str)
        .isin(
            remaining_local_ids
        )
    ]
    .copy()
)


df_local_completion = (
    df_local_completion
    .sort_values(
        "Genome ID"
    )
    .reset_index(
        drop=True
    )
)


LOCAL_COMPLETION_MANIFEST_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "amrfinder_local_completion_manifest.csv"
)


df_local_completion.to_csv(
    LOCAL_COMPLETION_MANIFEST_PATH,
    index=False
)


# =========================================================
# 8. Save unresolved audit
# =========================================================

UNRESOLVED_AUDIT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "ncbi_unresolved_genomes_audit.csv"
)


df_unresolved.to_csv(
    UNRESOLVED_AUDIT_PATH,
    index=False
)


# =========================================================
# 9. Final summary
# =========================================================

print()
print("=" * 70)
print("LOCAL AMRFINDER COMPLETION PLAN")
print("=" * 70)

print(
    f"Unresolved NCBI genomes        : "
    f"{len(unresolved_ids):,}"
)

print(
    f"Existing local outputs overall : "
    f"{len(existing_output_ids):,}"
)

print(
    f"Unresolved already processed   : "
    f"{len(already_completed_unresolved):,}"
)

print(
    f"Still requiring local screening: "
    f"{len(remaining_local_ids):,}"
)

print()

print(
    f"Estimated local runtime        : "
    f"{estimated_hours:.2f} h"
)

print(
    f"Original full-cohort estimate  : "
    f"{full_cohort_hours:.1f} h"
)

print(
    f"Estimated compute reduction    : "
    f"{time_reduction:.1%}"
)

print()

print(
    "Local completion manifest:"
)

print(
    LOCAL_COMPLETION_MANIFEST_PATH
)

print()

print(
    "PASS: NCBI coverage was audited and the "
    "remaining local AMRFinder workload was defined."
)

NCBI COVERAGE BIAS AUDIT
Resolved genomes   : 3,851
Unresolved genomes : 382

Resolved phenotype distribution:
Resistant Phenotype
Susceptible    2361
Resistant      1490
Name: count, dtype: int64

Unresolved phenotype distribution:
Resistant Phenotype
Susceptible    269
Resistant      113
Name: count, dtype: int64

NCBI resolution rate by phenotype:


,Total,Resolved,Unresolved,Resolution Rate
Resistant Phenotype,,,,
Resistant,1603,1490,113,0.929507
Susceptible,2630,2361,269,0.897719



LOCAL AMRFINDER COMPLETION PLAN
Unresolved NCBI genomes        : 382
Existing local outputs overall : 14
Unresolved already processed   : 0
Still requiring local screening: 382

Estimated local runtime        : 3.74 h
Original full-cohort estimate  : 41.4 h
Estimated compute reduction    : 91.0%

Local completion manifest:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/amrfinder_local_completion_manifest.csv

PASS: NCBI coverage was audited and the remaining local AMRFinder workload was defined.


# Cell 13 — Restore and Verify the Local AMRFinderPlus Runtime

In [2]:
import os
import shutil
import subprocess
import pandas as pd


# =========================================================
# 1. Expected reference configuration
# =========================================================

EXPECTED_AMRFINDER_VERSION = "4.2.7"
EXPECTED_DB_VERSION = "2026-08-07.1"

AMRFINDER_ORGANISM = (
    "Klebsiella_pneumoniae"
)


# =========================================================
# 2. Persistent runtime verification record
# =========================================================

RUNTIME_RECORD_PATH = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "amrfinder_file04_runtime_verified.csv"
)


# =========================================================
# 3. Check whether AMRFinder is already available
# =========================================================

amrfinder_path = shutil.which(
    "amrfinder"
)


if amrfinder_path is None:

    print(
        "AMRFinderPlus is not available "
        "in the current Colab runtime."
    )

    print()
    print(
        "Installing AMRFinderPlus 4.2.7..."
    )


    # -----------------------------------------------------
    # Install micromamba if necessary
    # -----------------------------------------------------

    MICROMAMBA_BIN = (
        "/content/micromamba"
    )


    if not os.path.exists(
        MICROMAMBA_BIN
    ):

        subprocess.run(
            [
                "bash",
                "-lc",
                (
                    "curl -Ls "
                    "https://micro.mamba.pm/api/"
                    "micromamba/linux-64/latest "
                    "| tar -xvj "
                    "bin/micromamba "
                    "--strip-components=1 "
                    "-C /usr/local/bin"
                )
            ],
            check=True
        )


    # -----------------------------------------------------
    # Create AMRFinder environment
    # -----------------------------------------------------

    AMRFINDER_ENV = (
        "/content/amrfinder_env"
    )


    subprocess.run(
        [
            MICROMAMBA_BIN,
            "create",
            "-y",
            "-p",
            AMRFINDER_ENV,
            "-c",
            "conda-forge",
            "-c",
            "bioconda",
            "ncbi-amrfinderplus=4.2.7"
        ],
        check=True
    )


    # Add environment binaries to PATH
    os.environ["PATH"] = (
        os.path.join(
            AMRFINDER_ENV,
            "bin"
        )
        + os.pathsep
        + os.environ["PATH"]
    )


    amrfinder_path = shutil.which(
        "amrfinder"
    )


# =========================================================
# 4. Verify executable
# =========================================================

assert amrfinder_path is not None, (
    "AMRFinderPlus installation failed."
)


version_result = subprocess.run(
    [
        amrfinder_path,
        "--version"
    ],
    capture_output=True,
    text=True,
    check=True
)


version_text = (
    version_result.stdout.strip()
    or version_result.stderr.strip()
)


print()
print(
    "AMRFinder executable:"
)

print(
    amrfinder_path
)

print()
print(
    "AMRFinder version output:"
)

print(
    version_text
)


assert EXPECTED_AMRFINDER_VERSION in version_text, (
    f"Expected AMRFinderPlus "
    f"{EXPECTED_AMRFINDER_VERSION}, "
    f"but received: {version_text}"
)


# =========================================================
# 5. Restore/update AMRFinder database
# =========================================================

print()
print(
    "Checking AMRFinderPlus database..."
)


update_result = subprocess.run(
    [
        amrfinder_path,
        "-u"
    ],
    capture_output=True,
    text=True
)


update_text = (
    update_result.stdout
    + "\n"
    + update_result.stderr
)


print(
    update_text[-4000:]
)


if update_result.returncode != 0:

    raise RuntimeError(
        "AMRFinderPlus database update failed."
    )


# =========================================================
# 6. Retrieve installed database version
# =========================================================

db_check = subprocess.run(
    [
        amrfinder_path,
        "--database_version"
    ],
    capture_output=True,
    text=True
)


db_version_text = (
    db_check.stdout.strip()
    or db_check.stderr.strip()
)


print()
print(
    "Database version:"
)

print(
    db_version_text
)


assert EXPECTED_DB_VERSION in db_version_text, (
    f"Expected AMRFinder database "
    f"{EXPECTED_DB_VERSION}, "
    f"but found: {db_version_text}"
)


# =========================================================
# 7. Verify local-completion manifest
# =========================================================

LOCAL_COMPLETION_MANIFEST_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "amrfinder_local_completion_manifest.csv"
)


assert os.path.exists(
    LOCAL_COMPLETION_MANIFEST_PATH
)


df_local_completion = pd.read_csv(
    LOCAL_COMPLETION_MANIFEST_PATH,
    dtype={
        "Genome ID": str,
        "Taxon ID": str
    }
)


assert len(
    df_local_completion
) == 382


assert (
    df_local_completion[
        "Genome ID"
    ]
    .nunique()
    == 382
)


# =========================================================
# 8. Save runtime verification
# =========================================================

df_runtime_record = pd.DataFrame(
    [
        {
            "AMRFinderPlus Version":
                EXPECTED_AMRFINDER_VERSION,

            "Database Version":
                EXPECTED_DB_VERSION,

            "Organism":
                AMRFINDER_ORGANISM,

            "Screening Mode":
                "NUCLEOTIDE",

            "Local Completion Genomes":
                len(
                    df_local_completion
                ),

            "Executable":
                amrfinder_path
        }
    ]
)


df_runtime_record.to_csv(
    RUNTIME_RECORD_PATH,
    index=False
)


# =========================================================
# 9. Final summary
# =========================================================

print()
print("=" * 70)
print("AMRFINDERPLUS LOCAL COMPLETION RUNTIME")
print("=" * 70)

print(
    f"Software version     : "
    f"{EXPECTED_AMRFINDER_VERSION}"
)

print(
    f"Database version     : "
    f"{EXPECTED_DB_VERSION}"
)

print(
    f"Organism             : "
    f"{AMRFINDER_ORGANISM}"
)

print(
    f"Screening mode       : "
    f"NUCLEOTIDE"
)

print(
    f"Genomes to process   : "
    f"{len(df_local_completion):,}"
)

print()

print(
    "Runtime verification saved to:"
)

print(
    RUNTIME_RECORD_PATH
)

print()

print(
    "PASS: AMRFinderPlus runtime is ready "
    "for local completion screening."
)

AMRFinderPlus is not available in the current Colab runtime.

Installing AMRFinderPlus 4.2.7...

AMRFinder executable:
/content/amrfinder_env/bin/amrfinder

AMRFinder version output:
4.2.7

Checking AMRFinderPlus database...

Running: /content/amrfinder_env/bin/amrfinder -u
The number of threads cannot be greater than 2 on this computer
The current number of threads is 4, reducing to 2
Software directory: /content/amrfinder_env/bin/
Software version: 4.2.7
Reverting to hard coded directory: /content/amrfinder_env/share/amrfinderplus/data/latest
Running: /content/amrfinder_env/bin/amrfinder_update -d /content/amrfinder_env/share/amrfinderplus/data
Looking up the published databases at https://ftp.ncbi.nlm.nih.gov/pathogen/Antimicrobial_resistance/AMRFinderPlus/database/
Looking for the target directory: /content/amrfinder_env/share/amrfinderplus/data/2026-08-07.1/
Running: /content/amrfinder_env/bin/amrfinder_index /content/amrfinder_env/share/amrfinderplus/data/2026-08-07.1/
Indexing
a

# Cell 14 — Run Resume-Safe Local AMRFinderPlus Completion

In [3]:
from tqdm.auto import tqdm
import os
import time
import gzip
import shutil
import subprocess
import pandas as pd

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)


# =========================================================
# 1. Runtime configuration
# =========================================================

AMRFINDER_BIN = (
    "/content/amrfinder_env/bin/amrfinder"
)

AMRFINDER_ORGANISM = (
    "Klebsiella_pneumoniae"
)

MAX_WORKERS = 2


assert os.path.exists(
    AMRFINDER_BIN
), "AMRFinderPlus executable is missing."


# =========================================================
# 2. Persistent paths
# =========================================================

FASTA_DIR = os.path.join(
    DRIVE_ROOT,
    "data",
    "genomes",
    "fasta_gz"
)

RAW_AMRFINDER_DIR = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "known_amr",
    "raw_amrfinder"
)

LOCAL_COMPLETION_MANIFEST_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "amrfinder_local_completion_manifest.csv"
)

LOCAL_COMPLETION_AUDIT_PATH = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "amrfinder_local_completion_audit.csv"
)


os.makedirs(
    RAW_AMRFINDER_DIR,
    exist_ok=True
)


# Temporary local storage only
LOCAL_TMP_DIR = (
    "/content/amrfinder_local_completion"
)

os.makedirs(
    LOCAL_TMP_DIR,
    exist_ok=True
)


# =========================================================
# 3. Reload completion manifest
# =========================================================

df_local_completion = pd.read_csv(
    LOCAL_COMPLETION_MANIFEST_PATH,
    dtype={
        "Genome ID": str,
        "Taxon ID": str
    }
)


df_local_completion[
    "Genome ID"
] = (
    df_local_completion[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


assert len(
    df_local_completion
) == 382


assert (
    df_local_completion[
        "Genome ID"
    ]
    .nunique()
    == 382
)


# =========================================================
# 4. Resolve FASTA paths
# =========================================================

def resolve_fasta_path(row):

    # Prefer the persistent path recorded in File 03
    if (
        "FASTA Path"
        in row.index
        and pd.notna(
            row["FASTA Path"]
        )
    ):

        path = str(
            row["FASTA Path"]
        ).strip()

        if os.path.exists(
            path
        ):

            return path


    # Fallback to standardized File 03 location
    genome_id = str(
        row["Genome ID"]
    )

    fallback = os.path.join(
        FASTA_DIR,
        f"{genome_id}.fna.gz"
    )


    return fallback


df_local_completion[
    "Resolved FASTA Path"
] = (
    df_local_completion.apply(
        resolve_fasta_path,
        axis=1
    )
)


missing_fasta = (
    df_local_completion[
        ~df_local_completion[
            "Resolved FASTA Path"
        ]
        .apply(
            os.path.exists
        )
    ]
)


assert missing_fasta.empty, (
    f"{len(missing_fasta)} FASTA files are missing."
)


# =========================================================
# 5. Detect already completed outputs
# =========================================================

def output_path_for(
    genome_id
):

    return os.path.join(
        RAW_AMRFINDER_DIR,
        f"{genome_id}.amrfinder.tsv"
    )


def valid_existing_output(
    genome_id
):

    path = output_path_for(
        genome_id
    )

    return (
        os.path.exists(path)
        and os.path.getsize(path) > 0
    )


already_complete_ids = {
    genome_id
    for genome_id
    in df_local_completion[
        "Genome ID"
    ]
    if valid_existing_output(
        genome_id
    )
}


df_pending = (
    df_local_completion[
        ~df_local_completion[
            "Genome ID"
        ]
        .isin(
            already_complete_ids
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print("=" * 70)
print("LOCAL AMRFINDERPLUS COMPLETION")
print("=" * 70)

print(
    f"Target genomes          : "
    f"{len(df_local_completion):,}"
)

print(
    f"Already complete        : "
    f"{len(already_complete_ids):,}"
)

print(
    f"Remaining this run      : "
    f"{len(df_pending):,}"
)

print(
    f"Parallel workers        : "
    f"{MAX_WORKERS}"
)

print()


# =========================================================
# 6. AMRFinder environment
# =========================================================

run_env = os.environ.copy()

run_env[
    "CONDA_PREFIX"
] = (
    "/content/amrfinder_env"
)

run_env[
    "PATH"
] = (
    "/content/amrfinder_env/bin:"
    + run_env.get(
        "PATH",
        ""
    )
)


# =========================================================
# 7. Per-genome worker
# =========================================================

def run_amrfinder_genome(
    genome_id,
    fasta_gz_path
):

    start_time = time.time()

    output_path = output_path_for(
        genome_id
    )


    # -----------------------------------------------------
    # Resume-safe skip
    # -----------------------------------------------------

    if (
        os.path.exists(
            output_path
        )
        and os.path.getsize(
            output_path
        ) > 0
    ):

        return {
            "Genome ID":
                genome_id,

            "Status":
                "SKIP",

            "Runtime Seconds":
                0.0,

            "Output Path":
                output_path,

            "Error":
                None
        }


    local_fasta = os.path.join(
        LOCAL_TMP_DIR,
        f"{genome_id}.fna"
    )

    local_output = os.path.join(
        LOCAL_TMP_DIR,
        f"{genome_id}.amrfinder.tsv"
    )


    try:

        # -------------------------------------------------
        # Decompress FASTA from Drive to local disk
        # -------------------------------------------------

        with gzip.open(
            fasta_gz_path,
            "rb"
        ) as src:

            with open(
                local_fasta,
                "wb"
            ) as dst:

                shutil.copyfileobj(
                    src,
                    dst
                )


        # -------------------------------------------------
        # Run nucleotide-only AMRFinderPlus
        # -------------------------------------------------

        command = [
            AMRFINDER_BIN,
            "-n",
            local_fasta,
            "-O",
            AMRFINDER_ORGANISM,
            "--threads",
            "1",
            "-o",
            local_output
        ]


        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            env=run_env
        )


        if result.returncode != 0:

            raise RuntimeError(
                result.stderr[
                    -3000:
                ]
            )


        if (
            not os.path.exists(
                local_output
            )
            or os.path.getsize(
                local_output
            ) == 0
        ):

            raise RuntimeError(
                "AMRFinderPlus produced no output file."
            )


        # -------------------------------------------------
        # Atomic-ish persistent save to Drive
        # -------------------------------------------------

        drive_tmp = (
            output_path
            + ".tmp"
        )


        shutil.copy2(
            local_output,
            drive_tmp
        )


        os.replace(
            drive_tmp,
            output_path
        )


        elapsed = (
            time.time()
            - start_time
        )


        return {
            "Genome ID":
                genome_id,

            "Status":
                "PASS",

            "Runtime Seconds":
                round(
                    elapsed,
                    2
                ),

            "Output Path":
                output_path,

            "Error":
                None
        }


    except Exception as exc:

        elapsed = (
            time.time()
            - start_time
        )


        return {
            "Genome ID":
                genome_id,

            "Status":
                "FAIL",

            "Runtime Seconds":
                round(
                    elapsed,
                    2
                ),

            "Output Path":
                output_path,

            "Error":
                str(exc)
        }


    finally:

        for path in [
            local_fasta,
            local_output
        ]:

            if os.path.exists(
                path
            ):

                try:

                    os.remove(
                        path
                    )

                except Exception:

                    pass


# =========================================================
# 8. Load previous checkpoint if available
# =========================================================

if os.path.exists(
    LOCAL_COMPLETION_AUDIT_PATH
):

    df_previous_audit = pd.read_csv(
        LOCAL_COMPLETION_AUDIT_PATH,
        dtype={
            "Genome ID": str
        }
    )

else:

    df_previous_audit = pd.DataFrame()


session_results = []


# =========================================================
# 9. Run parallel local completion with lighter checkpointing
# =========================================================

CHECKPOINT_EVERY = 10

run_start = time.time()


def save_completion_checkpoint():
    """
    Save the audit table to Drive.
    Individual AMRFinder TSV outputs are still saved immediately
    by each worker; this only reduces repeated CSV writes.
    """

    if len(session_results) == 0:
        return


    df_session = pd.DataFrame(
        session_results
    )


    if not df_previous_audit.empty:

        df_checkpoint = pd.concat(
            [
                df_previous_audit,
                df_session
            ],
            ignore_index=True
        )

    else:

        df_checkpoint = (
            df_session.copy()
        )


    df_checkpoint = (
        df_checkpoint
        .drop_duplicates(
            subset=[
                "Genome ID"
            ],
            keep="last"
        )
    )


    df_checkpoint.to_csv(
        LOCAL_COMPLETION_AUDIT_PATH,
        index=False
    )


if len(df_pending) > 0:

    successful_session = 0
    failed_session = 0
    processed_session = 0


    progress_bar = tqdm(
        total=len(df_local_completion),
        initial=len(already_complete_ids),
        desc="AMRFinder",
        unit="genome",
        dynamic_ncols=True,
        smoothing=0.2,
        mininterval=1.0
    )


    progress_bar.set_postfix(
        {
            "PASS":
                len(already_complete_ids),

            "FAIL":
                0,

            "left":
                len(df_pending)
        }
    )


    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {}


        # -------------------------------------------------
        # Submit only unfinished genomes
        # -------------------------------------------------

        for _, row in df_pending.iterrows():

            genome_id = row[
                "Genome ID"
            ]

            fasta_path = row[
                "Resolved FASTA Path"
            ]


            future = executor.submit(
                run_amrfinder_genome,
                genome_id,
                fasta_path
            )


            futures[
                future
            ] = genome_id


        # -------------------------------------------------
        # Collect completed jobs
        # -------------------------------------------------

        for future in as_completed(
            futures
        ):

            genome_id = futures[
                future
            ]


            try:

                result = future.result()

            except Exception as exc:

                result = {
                    "Genome ID":
                        genome_id,

                    "Status":
                        "FAIL",

                    "Runtime Seconds":
                        None,

                    "Output Path":
                        None,

                    "Error":
                        str(exc)
                }


            session_results.append(
                result
            )


            processed_session += 1


            if result[
                "Status"
            ] in {
                "PASS",
                "SKIP"
            }:

                successful_session += 1

            else:

                failed_session += 1


            # -------------------------------------------------
            # Write audit CSV only every 10 completed jobs
            # or immediately if a failure occurs
            # -------------------------------------------------

            if (
                processed_session
                % CHECKPOINT_EVERY
                == 0
                or result[
                    "Status"
                ] == "FAIL"
            ):

                save_completion_checkpoint()


            # -------------------------------------------------
            # Update live progress
            # -------------------------------------------------

            progress_bar.update(
                1
            )


            progress_bar.set_postfix(
                {
                    "PASS":
                        (
                            len(
                                already_complete_ids
                            )
                            + successful_session
                        ),

                    "FAIL":
                        failed_session,

                    "left":
                        (
                            len(df_pending)
                            - processed_session
                        )
                }
            )


    # -----------------------------------------------------
    # Final audit flush
    # -----------------------------------------------------

    save_completion_checkpoint()

    progress_bar.close()


else:

    print(
        "All 382 target genomes already have "
        "completed AMRFinder outputs."
    )


# =========================================================
# 10. Final integrity audit
# =========================================================

final_complete_ids = {
    genome_id
    for genome_id
    in df_local_completion[
        "Genome ID"
    ]
    if valid_existing_output(
        genome_id
    )
}


remaining_ids = set(
    df_local_completion[
        "Genome ID"
    ]
) - final_complete_ids


final_hours = (
    time.time()
    - run_start
) / 3600


print()
print("=" * 70)
print("LOCAL COMPLETION FINAL STATUS")
print("=" * 70)

print(
    f"Target genomes           : "
    f"{len(df_local_completion):,}"
)

print(
    f"Completed outputs        : "
    f"{len(final_complete_ids):,}"
)

print(
    f"Still missing            : "
    f"{len(remaining_ids):,}"
)

print(
    f"Current-session wall time: "
    f"{final_hours:.2f} h"
)


if remaining_ids:

    print()
    print(
        "Remaining Genome IDs:"
    )

    print(
        sorted(
            remaining_ids
        )[:50]
    )

    print()
    print(
        "PARTIAL: Some genomes remain. "
        "Re-running Cell 14 will automatically "
        "skip completed genomes and resume."
    )

else:

    print()
    print(
        "PASS: Local AMRFinderPlus completion "
        "is finished for all 382 unresolved genomes."
    )


print()
print(
    "Checkpoint audit:"
)

print(
    LOCAL_COMPLETION_AUDIT_PATH
)

LOCAL AMRFINDERPLUS COMPLETION
Target genomes          : 382
Already complete        : 382
Remaining this run      : 0
Parallel workers        : 2

All 382 target genomes already have completed AMRFinder outputs.

LOCAL COMPLETION FINAL STATUS
Target genomes           : 382
Completed outputs        : 382
Still missing            : 0
Current-session wall time: 0.00 h

PASS: Local AMRFinderPlus completion is finished for all 382 unresolved genomes.

Checkpoint audit:
/content/drive/MyDrive/AMR-Genome-ML/checkpoints/amrfinder_local_completion_audit.csv


# Cell 15 — Harmonize and Merge Known-AMR Determinants

In [4]:
import os
import re
import glob
import pandas as pd


# =========================================================
# 1. Persistent input paths
# =========================================================

NCBI_RESOLUTION_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "ncbi_full_cohort_amr_resolution.csv"
)

LOCAL_COMPLETION_MANIFEST_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "amrfinder_local_completion_manifest.csv"
)

RAW_AMRFINDER_DIR = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "known_amr",
    "raw_amrfinder"
)


# =========================================================
# 2. Persistent output paths
# =========================================================

KNOWN_AMR_LONG_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_determinants_long.csv"
)

KNOWN_AMR_COVERAGE_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_genome_coverage.csv"
)

KNOWN_AMR_HARMONIZATION_AUDIT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_harmonization_audit.csv"
)


# =========================================================
# 3. Reload NCBI resolution table
# =========================================================

df_ncbi_resolution_15 = pd.read_csv(
    NCBI_RESOLUTION_PATH,
    dtype={
        "Genome ID": str
    }
)


df_ncbi_resolution_15[
    "Genome ID"
] = (
    df_ncbi_resolution_15[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


# Handle boolean column safely after CSV reload
if (
    df_ncbi_resolution_15[
        "Usable NCBI AMR"
    ].dtype != bool
):

    df_ncbi_resolution_15[
        "Usable NCBI AMR"
    ] = (
        df_ncbi_resolution_15[
            "Usable NCBI AMR"
        ]
        .astype(str)
        .str.lower()
        .eq("true")
    )


df_ncbi_usable_15 = (
    df_ncbi_resolution_15[
        df_ncbi_resolution_15[
            "Usable NCBI AMR"
        ]
    ]
    .copy()
)


assert len(
    df_ncbi_usable_15
) == 3851, (
    f"Expected 3,851 usable NCBI genomes, "
    f"found {len(df_ncbi_usable_15):,}."
)


assert (
    df_ncbi_usable_15[
        "Genome ID"
    ]
    .nunique()
    == 3851
)


# =========================================================
# 4. Reload local-completion cohort
# =========================================================

df_local_manifest_15 = pd.read_csv(
    LOCAL_COMPLETION_MANIFEST_PATH,
    dtype={
        "Genome ID": str,
        "Taxon ID": str
    }
)


df_local_manifest_15[
    "Genome ID"
] = (
    df_local_manifest_15[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


local_genome_ids = set(
    df_local_manifest_15[
        "Genome ID"
    ]
)


assert len(
    local_genome_ids
) == 382


# =========================================================
# 5. Verify source assignment
# =========================================================

ncbi_genome_ids = set(
    df_ncbi_usable_15[
        "Genome ID"
    ]
)


assert ncbi_genome_ids.isdisjoint(
    local_genome_ids
), (
    "A genome was assigned to both NCBI and local AMR sources."
)


combined_source_ids = (
    ncbi_genome_ids
    | local_genome_ids
)


assert len(
    combined_source_ids
) == 4233, (
    f"Expected source coverage for 4,233 genomes, "
    f"found {len(combined_source_ids):,}."
)


# =========================================================
# 6. NCBI genotype parser
# =========================================================

def parse_ncbi_genotype_token(token):

    if pd.isna(token):
        return None, None

    token = str(
        token
    ).strip().strip(
        "\"'"
    )

    if not token:
        return None, None


    # ---------------------------------------------
    # Preserve reporting suffix before removing it
    #
    # Example:
    # emrD=HMM -> determinant=emrD, evidence=HMM
    # ---------------------------------------------

    match = re.search(
        r"=([A-Za-z0-9_+\-]+)$",
        token
    )


    evidence_suffix = (
        match.group(1).upper()
        if match
        else None
    )


    determinant = re.sub(
        r"=[A-Za-z0-9_+\-]+$",
        "",
        token
    ).strip()


    if not determinant:
        return None, evidence_suffix


    return determinant, evidence_suffix


# =========================================================
# 7. Parse NCBI AMR genotypes
# =========================================================

ncbi_records = []

ncbi_hmm_excluded = []


for _, row in df_ncbi_usable_15.iterrows():

    genome_id = row[
        "Genome ID"
    ]

    genotype_string = row.get(
        "AMR Genotypes"
    )


    if (
        pd.isna(genotype_string)
        or str(genotype_string).strip()
        in {
            "",
            "NULL",
            "None",
            "nan"
        }
    ):

        continue


    tokens = (
        str(
            genotype_string
        )
        .strip()
        .strip("\"")
        .split(",")
    )


    for raw_token in tokens:

        determinant, evidence_suffix = (
            parse_ncbi_genotype_token(
                raw_token
            )
        )


        if determinant is None:
            continue


        # ---------------------------------------------
        # Harmonization:
        # exclude HMM-only calls from NCBI combined mode
        # ---------------------------------------------

        if evidence_suffix == "HMM":

            ncbi_hmm_excluded.append(
                {
                    "Genome ID":
                        genome_id,

                    "Determinant":
                        determinant,

                    "Raw Token":
                        str(raw_token).strip(),

                    "Reason":
                        "NCBI_HMM_ONLY"
                }
            )

            continue


        ncbi_records.append(
            {
                "Genome ID":
                    genome_id,

                "Determinant":
                    determinant,

                "Source":
                    "NCBI",

                "Source Analysis Type":
                    row.get(
                        "Analysis Type"
                    ),

                "AMRFinder Version":
                    row.get(
                        "AMRFinder Version"
                    ),

                "Reference DB Version":
                    row.get(
                        "Reference DB Version"
                    ),

                "Subtype":
                    pd.NA,

                "Class":
                    pd.NA,

                "Method":
                    evidence_suffix
            }
        )


df_ncbi_determinants = pd.DataFrame(
    ncbi_records
)


# =========================================================
# 8. Read all 382 local AMRFinder outputs
# =========================================================

LOCAL_ALLOWED_SUBTYPES = {
    "AMR",
    "POINT",
    "POINT_DISRUPT"
}


local_records = []

local_file_audit = []


for genome_id in sorted(
    local_genome_ids
):

    path = os.path.join(
        RAW_AMRFINDER_DIR,
        f"{genome_id}.amrfinder.tsv"
    )


    if not os.path.exists(
        path
    ):

        local_file_audit.append(
            {
                "Genome ID":
                    genome_id,

                "Status":
                    "MISSING",

                "AMR Determinants":
                    0
            }
        )

        continue


    try:

        df_local_raw = pd.read_csv(
            path,
            sep="\t"
        )


        required_columns = {
            "Element symbol",
            "Type",
            "Subtype"
        }


        if not required_columns.issubset(
            df_local_raw.columns
        ):

            raise RuntimeError(
                "Unexpected AMRFinder output schema."
            )


        df_local_amr = (
            df_local_raw[
                df_local_raw[
                    "Type"
                ]
                .astype(str)
                .str.upper()
                .eq("AMR")
                &
                df_local_raw[
                    "Subtype"
                ]
                .astype(str)
                .str.upper()
                .isin(
                    LOCAL_ALLOWED_SUBTYPES
                )
            ]
            .copy()
        )


        determinant_count = 0


        for _, hit in df_local_amr.iterrows():

            determinant = hit.get(
                "Element symbol"
            )


            if (
                pd.isna(determinant)
                or not str(
                    determinant
                ).strip()
            ):

                continue


            determinant = str(
                determinant
            ).strip()


            local_records.append(
                {
                    "Genome ID":
                        genome_id,

                    "Determinant":
                        determinant,

                    "Source":
                        "LOCAL",

                    "Source Analysis Type":
                        "NUCLEOTIDE",

                    "AMRFinder Version":
                        "4.2.7",

                    "Reference DB Version":
                        "2026-08-07.1",

                    "Subtype":
                        hit.get(
                            "Subtype",
                            pd.NA
                        ),

                    "Class":
                        hit.get(
                            "Class",
                            pd.NA
                        ),

                    "Method":
                        hit.get(
                            "Method",
                            pd.NA
                        )
                }
            )


            determinant_count += 1


        local_file_audit.append(
            {
                "Genome ID":
                    genome_id,

                "Status":
                    "PASS",

                "AMR Determinants":
                    determinant_count
            }
        )


    except Exception as exc:

        local_file_audit.append(
            {
                "Genome ID":
                    genome_id,

                "Status":
                    "FAIL",

                "AMR Determinants":
                    0,

                "Error":
                    str(exc)
            }
        )


df_local_determinants = pd.DataFrame(
    local_records
)


df_local_file_audit = pd.DataFrame(
    local_file_audit
)


# =========================================================
# 9. Verify all 382 local files were readable
# =========================================================

failed_local = (
    df_local_file_audit[
        df_local_file_audit[
            "Status"
        ]
        .ne("PASS")
    ]
)


if not failed_local.empty:

    display(
        failed_local
    )

    raise RuntimeError(
        "One or more local AMRFinder outputs "
        "could not be loaded."
    )


assert len(
    df_local_file_audit
) == 382


# =========================================================
# 10. Merge NCBI and local determinants
# =========================================================

df_known_amr_long = pd.concat(
    [
        df_ncbi_determinants,
        df_local_determinants
    ],
    ignore_index=True
)


# =========================================================
# 11. Normalize determinant key
# =========================================================

df_known_amr_long[
    "Determinant"
] = (
    df_known_amr_long[
        "Determinant"
    ]
    .astype(str)
    .str.strip()
)


# Case-insensitive key while preserving biological punctuation
df_known_amr_long[
    "Determinant Key"
] = (
    df_known_amr_long[
        "Determinant"
    ]
    .str.lower()
)


# =========================================================
# 12. Remove duplicate genome-determinant occurrences
# =========================================================

before_dedup = len(
    df_known_amr_long
)


df_known_amr_long = (
    df_known_amr_long
    .sort_values(
        [
            "Genome ID",
            "Determinant Key"
        ]
    )
    .drop_duplicates(
        subset=[
            "Genome ID",
            "Determinant Key"
        ],
        keep="first"
    )
    .reset_index(
        drop=True
    )
)


duplicates_removed = (
    before_dedup
    - len(
        df_known_amr_long
    )
)


# =========================================================
# 13. Create a canonical display label for each determinant
# =========================================================

canonical_labels = (
    df_known_amr_long
    .groupby(
        "Determinant Key"
    )[
        "Determinant"
    ]
    .agg(
        lambda x:
        x.value_counts().index[0]
    )
)


df_known_amr_long[
    "Determinant"
] = (
    df_known_amr_long[
        "Determinant Key"
    ]
    .map(
        canonical_labels
    )
)


# =========================================================
# 14. Genome-level coverage table
# =========================================================

determinant_counts = (
    df_known_amr_long
    .groupby(
        "Genome ID"
    )
    .size()
    .rename(
        "Known AMR Determinants"
    )
)


df_known_amr_coverage = (
    df_feature_input[
        [
            "Genome ID"
        ]
    ]
    .copy()
)


df_known_amr_coverage[
    "Genome ID"
] = (
    df_known_amr_coverage[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


df_known_amr_coverage[
    "AMR Source"
] = (
    df_known_amr_coverage[
        "Genome ID"
    ]
    .map(
        lambda genome_id:
        (
            "NCBI"
            if genome_id
            in ncbi_genome_ids
            else "LOCAL"
        )
    )
)


df_known_amr_coverage = (
    df_known_amr_coverage
    .merge(
        determinant_counts,
        on="Genome ID",
        how="left"
    )
)


df_known_amr_coverage[
    "Known AMR Determinants"
] = (
    df_known_amr_coverage[
        "Known AMR Determinants"
    ]
    .fillna(0)
    .astype(int)
)


# =========================================================
# 15. Harmonization audit
# =========================================================

df_hmm_excluded = pd.DataFrame(
    ncbi_hmm_excluded
)


audit_records = [
    {
        "Metric":
            "Study genomes",

        "Value":
            4233
    },
    {
        "Metric":
            "NCBI genomes",

        "Value":
            len(
                ncbi_genome_ids
            )
    },
    {
        "Metric":
            "Local genomes",

        "Value":
            len(
                local_genome_ids
            )
    },
    {
        "Metric":
            "NCBI determinants retained",

        "Value":
            len(
                df_ncbi_determinants
            )
    },
    {
        "Metric":
            "NCBI HMM-only calls excluded",

        "Value":
            len(
                df_hmm_excluded
            )
    },
    {
        "Metric":
            "Local determinants retained",

        "Value":
            len(
                df_local_determinants
            )
    },
    {
        "Metric":
            "Duplicate genome-determinant rows removed",

        "Value":
            duplicates_removed
    },
    {
        "Metric":
            "Final genome-determinant occurrences",

        "Value":
            len(
                df_known_amr_long
            )
    },
    {
        "Metric":
            "Unique known-AMR determinants",

        "Value":
            df_known_amr_long[
                "Determinant Key"
            ].nunique()
    }
]


df_harmonization_audit = pd.DataFrame(
    audit_records
)


# =========================================================
# 16. Final integrity checks
# =========================================================

assert len(
    df_known_amr_coverage
) == 4233


assert (
    df_known_amr_coverage[
        "Genome ID"
    ]
    .nunique()
    == 4233
)


assert (
    df_known_amr_coverage[
        "AMR Source"
    ]
    .value_counts()
    .to_dict()
    == {
        "NCBI": 3851,
        "LOCAL": 382
    }
)


# =========================================================
# 17. Save persistent outputs
# =========================================================

df_known_amr_long.to_csv(
    KNOWN_AMR_LONG_PATH,
    index=False
)


df_known_amr_coverage.to_csv(
    KNOWN_AMR_COVERAGE_PATH,
    index=False
)


df_harmonization_audit.to_csv(
    KNOWN_AMR_HARMONIZATION_AUDIT_PATH,
    index=False
)


if not df_hmm_excluded.empty:

    df_hmm_excluded.to_csv(
        os.path.join(
            KNOWN_AMR_DIR,
            "ncbi_hmm_only_calls_excluded.csv"
        ),
        index=False
    )


# =========================================================
# 18. Summary
# =========================================================

print("=" * 70)
print("KNOWN-AMR HARMONIZATION")
print("=" * 70)

print(
    f"Study genomes                  : "
    f"{len(df_known_amr_coverage):,}"
)

print(
    f"NCBI source genomes            : "
    f"{len(ncbi_genome_ids):,}"
)

print(
    f"Local source genomes           : "
    f"{len(local_genome_ids):,}"
)

print()

print(
    f"NCBI determinant rows retained : "
    f"{len(df_ncbi_determinants):,}"
)

print(
    f"NCBI HMM-only calls excluded   : "
    f"{len(df_hmm_excluded):,}"
)

print(
    f"Local determinant rows retained: "
    f"{len(df_local_determinants):,}"
)

print()

print(
    f"Duplicate occurrences removed  : "
    f"{duplicates_removed:,}"
)

print(
    f"Final determinant occurrences  : "
    f"{len(df_known_amr_long):,}"
)

print(
    f"Unique determinant features    : "
    f"{df_known_amr_long['Determinant Key'].nunique():,}"
)

print()

print(
    "Determinants per genome:"
)

display(
    df_known_amr_coverage[
        "Known AMR Determinants"
    ]
    .describe()
    .to_frame()
)

print()

print(
    "Source distribution:"
)

print(
    df_known_amr_coverage[
        "AMR Source"
    ]
    .value_counts()
)

print()

print(
    "Unified determinant table saved to:"
)

print(
    KNOWN_AMR_LONG_PATH
)

print()

print(
    "PASS: NCBI and local AMRFinderPlus results were "
    "harmonized into a unified known-AMR determinant table "
    "for all 4,233 study genomes."
)

KNOWN-AMR HARMONIZATION
Study genomes                  : 4,233
NCBI source genomes            : 3,851
Local source genomes           : 382

NCBI determinant rows retained : 61,821
NCBI HMM-only calls excluded   : 82
Local determinant rows retained: 6,661

Duplicate occurrences removed  : 1,208
Final determinant occurrences  : 67,274
Unique determinant features    : 669

Determinants per genome:


,Known AMR Determinants
count,4233.000000
mean,15.892747
std,7.041521
min,0.000000
25%,10.000000
50%,17.000000
75%,20.000000
max,45.000000



Source distribution:
AMR Source
NCBI     3851
LOCAL     382
Name: count, dtype: int64

Unified determinant table saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/known_amr_determinants_long.csv

PASS: NCBI and local AMRFinderPlus results were harmonized into a unified known-AMR determinant table for all 4,233 study genomes.


# Cell 16 — Construct the Binary Known-AMR Feature Matrix

In [5]:
import os
import pandas as pd
import numpy as np


# =========================================================
# 1. Persistent paths
# =========================================================

KNOWN_AMR_LONG_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_determinants_long.csv"
)

X_KNOWN_AMR_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "X_known_amr_binary.csv"
)

FEATURE_METADATA_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_feature_metadata.csv"
)


# =========================================================
# 2. Reload harmonized determinant table
# =========================================================

df_known_amr_long_16 = pd.read_csv(
    KNOWN_AMR_LONG_PATH,
    dtype={
        "Genome ID": str,
        "Determinant": str,
        "Determinant Key": str
    }
)


df_known_amr_long_16[
    "Genome ID"
] = (
    df_known_amr_long_16[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


# =========================================================
# 3. Recover full 4,233-genome order
# =========================================================

study_genome_ids = (
    df_feature_input[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
    .tolist()
)


assert len(
    study_genome_ids
) == 4233


assert len(
    set(
        study_genome_ids
    )
) == 4233


# =========================================================
# 4. Construct presence table
# =========================================================

df_presence = (
    df_known_amr_long_16[
        [
            "Genome ID",
            "Determinant Key"
        ]
    ]
    .drop_duplicates()
    .assign(
        Present=1
    )
)


# =========================================================
# 5. Pivot into binary feature matrix
# =========================================================

X_known_amr = (
    df_presence
    .pivot(
        index="Genome ID",
        columns="Determinant Key",
        values="Present"
    )
    .fillna(0)
)


# Ensure all 4,233 study genomes are represented,
# including genomes with zero retained determinants.
X_known_amr = (
    X_known_amr
    .reindex(
        study_genome_ids,
        fill_value=0
    )
)


# Store binary values efficiently
X_known_amr = (
    X_known_amr
    .astype(
        np.uint8
    )
)


# =========================================================
# 6. Build canonical feature names
# =========================================================

canonical_feature_names = (
    df_known_amr_long_16[
        [
            "Determinant Key",
            "Determinant"
        ]
    ]
    .drop_duplicates(
        subset=[
            "Determinant Key"
        ]
    )
    .set_index(
        "Determinant Key"
    )[
        "Determinant"
    ]
    .to_dict()
)


# Rename normalized keys back to readable biological labels
X_known_amr = (
    X_known_amr
    .rename(
        columns=canonical_feature_names
    )
)


# =========================================================
# 7. Integrity checks
# =========================================================

assert X_known_amr.shape[0] == 4233, (
    f"Expected 4,233 rows, "
    f"found {X_known_amr.shape[0]:,}."
)


assert X_known_amr.shape[1] == 669, (
    f"Expected 669 known-AMR features, "
    f"found {X_known_amr.shape[1]:,}."
)


unique_values = set(
    np.unique(
        X_known_amr.values
    )
)


assert unique_values.issubset(
    {
        0,
        1
    }
), (
    f"Non-binary values detected: "
    f"{unique_values}"
)


assert (
    X_known_amr.index.nunique()
    == 4233
)


assert (
    X_known_amr.columns.nunique()
    == X_known_amr.shape[1]
)


# =========================================================
# 8. Feature prevalence metadata
# =========================================================

feature_counts = (
    X_known_amr
    .sum(
        axis=0
    )
    .astype(int)
)


df_feature_metadata = pd.DataFrame(
    {
        "Feature":
            feature_counts.index,

        "Genomes Present":
            feature_counts.values
    }
)


df_feature_metadata[
    "Prevalence"
] = (
    df_feature_metadata[
        "Genomes Present"
    ]
    / len(
        X_known_amr
    )
)


df_feature_metadata = (
    df_feature_metadata
    .sort_values(
        [
            "Genomes Present",
            "Feature"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


# =========================================================
# 9. Genome-level feature counts
# =========================================================

genome_feature_counts = (
    X_known_amr
    .sum(
        axis=1
    )
    .astype(int)
)


zero_feature_genomes = int(
    (
        genome_feature_counts
        == 0
    )
    .sum()
)


# =========================================================
# 10. Save matrix
# =========================================================

X_known_amr_export = (
    X_known_amr
    .reset_index()
    .rename(
        columns={
            "index":
                "Genome ID"
        }
    )
)


X_known_amr_export.to_csv(
    X_KNOWN_AMR_PATH,
    index=False
)


df_feature_metadata.to_csv(
    FEATURE_METADATA_PATH,
    index=False
)


# =========================================================
# 11. Summary
# =========================================================

print("=" * 70)
print("KNOWN-AMR BINARY FEATURE MATRIX")
print("=" * 70)

print(
    f"Study genomes          : "
    f"{X_known_amr.shape[0]:,}"
)

print(
    f"Known-AMR features     : "
    f"{X_known_amr.shape[1]:,}"
)

print(
    f"Matrix dimensions      : "
    f"{X_known_amr.shape[0]:,} x "
    f"{X_known_amr.shape[1]:,}"
)

print(
    f"Binary values          : "
    f"{sorted(unique_values)}"
)

print(
    f"Zero-feature genomes   : "
    f"{zero_feature_genomes:,}"
)

print()


print(
    "Features per genome:"
)

display(
    genome_feature_counts
    .describe()
    .to_frame(
        name="Feature Count"
    )
)


print()
print(
    "Most prevalent known-AMR features:"
)

display(
    df_feature_metadata
    .head(20)
)


print()
print(
    "Least prevalent known-AMR features:"
)

display(
    df_feature_metadata
    .tail(20)
)


print()
print(
    "Feature matrix saved to:"
)

print(
    X_KNOWN_AMR_PATH
)


print()
print(
    "Feature metadata saved to:"
)

print(
    FEATURE_METADATA_PATH
)


print()
print(
    "PASS: X_known_amr was created successfully "
    "for all 4,233 study genomes."
)

KNOWN-AMR BINARY FEATURE MATRIX
Study genomes          : 4,233
Known-AMR features     : 669
Matrix dimensions      : 4,233 x 669
Binary values          : [np.uint8(0), np.uint8(1)]
Zero-feature genomes   : 3

Features per genome:


,Feature Count
count,4233.000000
mean,15.892747
std,7.041521
min,0.000000
25%,10.000000
50%,17.000000
75%,20.000000
max,45.000000



Most prevalent known-AMR features:


,Feature,Genomes Present,Prevalence
0,fosA,4022,0.950154
1,emrD,3851,0.909757
2,oqxA,3839,0.906922
3,parC_S80I,2274,0.537208
4,oqxB,2213,0.522797
5,blaTEM-1,2014,0.475785
6,gyrA_S83I,1881,0.444366
7,blaCTX-M-15,1860,0.439405
8,sul1,1787,0.422159
9,blaSHV-11,1561,0.368769



Least prevalent known-AMR features:


,Feature,Genomes Present,Prevalence
649,ramR_Q171AfsTer11,1,0.000236
650,ramR_Q23Ter,1,0.000236
651,ramR_R172AfsTer55,1,0.000236
652,ramR_S137Ter,1,0.000236
653,ramR_S144RfsTer83,1,0.000236
654,ramR_S157Ter,1,0.000236
655,ramR_S29HfsTer20,1,0.000236
656,ramR_S29RfsTer39,1,0.000236
657,ramR_S29RfsTer41,1,0.000236
658,ramR_T69PfsTer51,1,0.000236



Feature matrix saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/X_known_amr_binary.csv

Feature metadata saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/known_amr_feature_metadata.csv

PASS: X_known_amr was created successfully for all 4,233 study genomes.


# Cell 17 — Audit Technical Source Effects in the Known-AMR Matrix

In [6]:
import os
import numpy as np
import pandas as pd


# =========================================================
# 1. Reload source assignment
# =========================================================

KNOWN_AMR_COVERAGE_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_genome_coverage.csv"
)

df_source_audit = pd.read_csv(
    KNOWN_AMR_COVERAGE_PATH,
    dtype={
        "Genome ID": str
    }
)


df_source_audit["Genome ID"] = (
    df_source_audit["Genome ID"]
    .astype(str)
    .str.strip()
)


assert len(df_source_audit) == 4233


# =========================================================
# 2. Align source labels to X_known_amr
# =========================================================

source_map = (
    df_source_audit
    .set_index("Genome ID")[
        "AMR Source"
    ]
)


source_labels = (
    source_map
    .reindex(
        X_known_amr.index
    )
)


assert source_labels.notna().all()


ncbi_mask = (
    source_labels
    .eq("NCBI")
    .values
)

local_mask = (
    source_labels
    .eq("LOCAL")
    .values
)


assert ncbi_mask.sum() == 3851
assert local_mask.sum() == 382


# =========================================================
# 3. Calculate feature prevalence by source
# =========================================================

ncbi_prevalence = (
    X_known_amr.loc[
        ncbi_mask
    ]
    .mean(axis=0)
)


local_prevalence = (
    X_known_amr.loc[
        local_mask
    ]
    .mean(axis=0)
)


df_source_feature_audit = pd.DataFrame(
    {
        "Feature":
            X_known_amr.columns,

        "NCBI Prevalence":
            ncbi_prevalence.values,

        "Local Prevalence":
            local_prevalence.values
    }
)


df_source_feature_audit[
    "Absolute Difference"
] = (
    df_source_feature_audit[
        "NCBI Prevalence"
    ]
    - df_source_feature_audit[
        "Local Prevalence"
    ]
).abs()


# =========================================================
# 4. Count occurrences in each source
# =========================================================

df_source_feature_audit[
    "NCBI Count"
] = (
    X_known_amr.loc[
        ncbi_mask
    ]
    .sum(axis=0)
    .values
    .astype(int)
)


df_source_feature_audit[
    "Local Count"
] = (
    X_known_amr.loc[
        local_mask
    ]
    .sum(axis=0)
    .values
    .astype(int)
)


# =========================================================
# 5. Flag extreme source-specific features
# =========================================================

df_source_feature_audit[
    "Source Exclusive"
] = (
    (
        df_source_feature_audit[
            "NCBI Count"
        ].eq(0)
    )
    |
    (
        df_source_feature_audit[
            "Local Count"
        ].eq(0)
    )
)


df_source_feature_audit[
    "Extreme Difference"
] = (
    df_source_feature_audit[
        "Absolute Difference"
    ]
    >= 0.80
)


# =========================================================
# 6. Explicitly flag emrD based on paired pilot evidence
# =========================================================

df_source_feature_audit[
    "Pilot-confirmed Technical Artifact"
] = (
    df_source_feature_audit[
        "Feature"
    ]
    .astype(str)
    .str.lower()
    .eq("emrd")
)


# =========================================================
# 7. Sort suspicious features
# =========================================================

df_source_feature_audit = (
    df_source_feature_audit
    .sort_values(
        [
            "Pilot-confirmed Technical Artifact",
            "Absolute Difference"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)


# =========================================================
# 8. Summary
# =========================================================

print("=" * 70)
print("KNOWN-AMR SOURCE-EFFECT AUDIT")
print("=" * 70)

print(
    f"NCBI genomes              : "
    f"{ncbi_mask.sum():,}"
)

print(
    f"Local genomes             : "
    f"{local_mask.sum():,}"
)

print(
    f"Features audited          : "
    f"{len(df_source_feature_audit):,}"
)

print()

print(
    f"Source-exclusive features : "
    f"{df_source_feature_audit['Source Exclusive'].sum():,}"
)

print(
    f"Extreme prevalence gaps   : "
    f"{df_source_feature_audit['Extreme Difference'].sum():,}"
)

print()


print(
    "Features with the largest NCBI/local prevalence differences:"
)

display(
    df_source_feature_audit[
        [
            "Feature",
            "NCBI Count",
            "Local Count",
            "NCBI Prevalence",
            "Local Prevalence",
            "Absolute Difference",
            "Source Exclusive",
            "Pilot-confirmed Technical Artifact"
        ]
    ]
    .head(30)
)


# =========================================================
# 9. Inspect emrD specifically
# =========================================================

df_emrd_audit = (
    df_source_feature_audit[
        df_source_feature_audit[
            "Feature"
        ]
        .astype(str)
        .str.lower()
        .eq("emrd")
    ]
)


print()
print("emrD audit:")

display(
    df_emrd_audit
)


# =========================================================
# 10. Save audit
# =========================================================

SOURCE_EFFECT_AUDIT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_source_effect_audit.csv"
)


df_source_feature_audit.to_csv(
    SOURCE_EFFECT_AUDIT_PATH,
    index=False
)


print()
print(
    "Source-effect audit saved to:"
)

print(
    SOURCE_EFFECT_AUDIT_PATH
)

print()

print(
    "PASS: Known-AMR features were audited for "
    "technical NCBI-versus-local source effects."
)

KNOWN-AMR SOURCE-EFFECT AUDIT
NCBI genomes              : 3,851
Local genomes             : 382
Features audited          : 669

Source-exclusive features : 502
Extreme prevalence gaps   : 1

Features with the largest NCBI/local prevalence differences:


,Feature,NCBI Count,Local Count,NCBI Prevalence,Local Prevalence,Absolute Difference,Source Exclusive,Pilot-confirmed Technical Artifact
0,emrD,3851,0,1.000000,0.000000,1.000000,True,True
1,blaCTX-M-15,1580,280,0.410283,0.732984,0.322701,False,False
2,armA,83,131,0.021553,0.342932,0.321379,False,False
3,qnrB1,561,171,0.145676,0.447644,0.301968,False,False
4,sul2,1178,231,0.305895,0.604712,0.298817,False,False
5,blaOXA-1,835,190,0.216827,0.497382,0.280555,False,False
6,catB3,899,193,0.233446,0.505236,0.271790,False,False
7,aph(3'')-Ib,1181,197,0.306674,0.515707,0.209033,False,False
8,msr(E),84,88,0.021813,0.230366,0.208554,False,False
9,mph(E),85,88,0.022072,0.230366,0.208294,False,False



emrD audit:


,Feature,NCBI Prevalence,Local Prevalence,Absolute Difference,NCBI Count,Local Count,Source Exclusive,Extreme Difference,Pilot-confirmed Technical Artifact
0,emrD,1.0,0.0,1.0,3851,0,True,True,True



Source-effect audit saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/known_amr_source_effect_audit.csv

PASS: Known-AMR features were audited for technical NCBI-versus-local source effects.


# Cell 18 — Finalize the Harmonized Known-AMR Feature Matrix

In [7]:
import os
import pandas as pd
import numpy as np


# =========================================================
# 1. Persistent paths
# =========================================================

X_KNOWN_AMR_RAW_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "X_known_amr_binary.csv"
)

X_KNOWN_AMR_FINAL_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "X_known_amr_final.csv"
)

FINAL_FEATURE_METADATA_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_feature_metadata_final.csv"
)

FINAL_HARMONIZATION_DECISION_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_final_harmonization_decision.csv"
)


# =========================================================
# 2. Reload binary matrix
# =========================================================

df_x_raw = pd.read_csv(
    X_KNOWN_AMR_RAW_PATH,
    dtype={
        "Genome ID": str
    }
)


df_x_raw[
    "Genome ID"
] = (
    df_x_raw[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


assert df_x_raw.shape == (
    4233,
    670
), (
    f"Expected 4,233 genomes + 669 features, "
    f"found {df_x_raw.shape}."
)


# =========================================================
# 3. Identify technical artifact
# =========================================================

feature_columns = [
    col
    for col in df_x_raw.columns
    if col != "Genome ID"
]


emrd_columns = [
    col
    for col in feature_columns
    if str(col)
    .strip()
    .lower()
    == "emrd"
]


assert len(emrd_columns) == 1, (
    f"Expected exactly one emrD feature, "
    f"found {emrd_columns}."
)


technical_artifact = (
    emrd_columns[0]
)


print(
    "Technical artifact identified:",
    technical_artifact
)


# =========================================================
# 4. Remove emrD
# =========================================================

df_x_final = (
    df_x_raw
    .drop(
        columns=[
            technical_artifact
        ]
    )
    .copy()
)


# =========================================================
# 5. Final matrix integrity
# =========================================================

final_feature_columns = [
    col
    for col in df_x_final.columns
    if col != "Genome ID"
]


assert len(
    df_x_final
) == 4233


assert (
    df_x_final[
        "Genome ID"
    ]
    .nunique()
    == 4233
)


assert len(
    final_feature_columns
) == 668


unique_values = set(
    np.unique(
        df_x_final[
            final_feature_columns
        ].values
    )
)


assert unique_values.issubset(
    {
        0,
        1
    }
)


assert not any(
    str(col).lower()
    == "emrd"
    for col in final_feature_columns
)


# =========================================================
# 6. Final feature metadata
# =========================================================

feature_counts = (
    df_x_final[
        final_feature_columns
    ]
    .sum(
        axis=0
    )
    .astype(int)
)


df_final_feature_metadata = pd.DataFrame(
    {
        "Feature":
            feature_counts.index,

        "Genomes Present":
            feature_counts.values
    }
)


df_final_feature_metadata[
    "Prevalence"
] = (
    df_final_feature_metadata[
        "Genomes Present"
    ]
    / len(
        df_x_final
    )
)


df_final_feature_metadata = (
    df_final_feature_metadata
    .sort_values(
        [
            "Genomes Present",
            "Feature"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


# =========================================================
# 7. Record harmonization decision
# =========================================================

df_final_decision = pd.DataFrame(
    [
        {
            "Feature":
                "emrD",

            "Decision":
                "EXCLUDE",

            "Reason":
                (
                    "Pilot-confirmed technical source marker; "
                    "present in all NCBI-derived genomes and "
                    "absent from all locally screened genomes."
                ),

            "Phenotype Used for Decision":
                False
        }
    ]
)


# =========================================================
# 8. Save finalized outputs
# =========================================================

df_x_final.to_csv(
    X_KNOWN_AMR_FINAL_PATH,
    index=False
)


df_final_feature_metadata.to_csv(
    FINAL_FEATURE_METADATA_PATH,
    index=False
)


df_final_decision.to_csv(
    FINAL_HARMONIZATION_DECISION_PATH,
    index=False
)


# =========================================================
# 9. Final summary
# =========================================================

print()
print("=" * 70)
print("FINAL KNOWN-AMR FEATURE MATRIX")
print("=" * 70)

print(
    f"Study genomes             : "
    f"{len(df_x_final):,}"
)

print(
    f"Initial AMR features      : "
    f"{len(feature_columns):,}"
)

print(
    f"Technical artifacts removed: "
    f"1"
)

print(
    f"Final AMR features        : "
    f"{len(final_feature_columns):,}"
)

print(
    f"Final matrix dimensions   : "
    f"{len(df_x_final):,} x "
    f"{len(final_feature_columns):,}"
)

print(
    f"Binary values             : "
    f"{sorted(unique_values)}"
)

print()

print(
    "Removed feature:"
)

display(
    df_final_decision
)

print()

print(
    "Final matrix saved to:"
)

print(
    X_KNOWN_AMR_FINAL_PATH
)

print()

print(
    "PASS: The harmonized X_known_amr matrix "
    "was finalized with 668 genomic features "
    "across all 4,233 study genomes."
)

Technical artifact identified: emrD

FINAL KNOWN-AMR FEATURE MATRIX
Study genomes             : 4,233
Initial AMR features      : 669
Technical artifacts removed: 1
Final AMR features        : 668
Final matrix dimensions   : 4,233 x 668
Binary values             : [np.int64(0), np.int64(1)]

Removed feature:


,Feature,Decision,Reason,Phenotype Used for Decision
0,emrD,EXCLUDE,Pilot-confirmed technical source marker; prese...,False



Final matrix saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/X_known_amr_final.csv

PASS: The harmonized X_known_amr matrix was finalized with 668 genomic features across all 4,233 study genomes.


# Cell 19 — Final Integrity Audit of the Known-AMR Feature Matrix

In [8]:
import os
import numpy as np
import pandas as pd


# =========================================================
# 1. Persistent paths
# =========================================================

X_KNOWN_AMR_FINAL_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "X_known_amr_final.csv"
)

FINAL_AUDIT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_final_integrity_audit.csv"
)


# =========================================================
# 2. Reload finalized matrix
# =========================================================

df_x_final_19 = pd.read_csv(
    X_KNOWN_AMR_FINAL_PATH,
    dtype={
        "Genome ID": str
    }
)


df_x_final_19[
    "Genome ID"
] = (
    df_x_final_19[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


feature_columns = [
    col
    for col in df_x_final_19.columns
    if col != "Genome ID"
]


# =========================================================
# 3. Cohort integrity
# =========================================================

n_genomes = len(
    df_x_final_19
)

n_unique_genomes = (
    df_x_final_19[
        "Genome ID"
    ]
    .nunique()
)

n_features = len(
    feature_columns
)


expected_ids = set(
    df_feature_input[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


matrix_ids = set(
    df_x_final_19[
        "Genome ID"
    ]
)


missing_genomes = (
    expected_ids
    - matrix_ids
)

unexpected_genomes = (
    matrix_ids
    - expected_ids
)


# =========================================================
# 4. Missing-value audit
# =========================================================

missing_values = int(
    df_x_final_19[
        feature_columns
    ]
    .isna()
    .sum()
    .sum()
)


# =========================================================
# 5. Binary-value audit
# =========================================================

unique_values = set(
    np.unique(
        df_x_final_19[
            feature_columns
        ].values
    )
)


binary_valid = (
    unique_values
    .issubset(
        {
            0,
            1
        }
    )
)


# =========================================================
# 6. Constant-feature audit
# =========================================================

feature_variation = (
    df_x_final_19[
        feature_columns
    ]
    .nunique(
        axis=0
    )
)


constant_features = (
    feature_variation[
        feature_variation
        <= 1
    ]
    .index
    .tolist()
)


# =========================================================
# 7. Zero-feature genomes
# =========================================================

genome_feature_counts = (
    df_x_final_19[
        feature_columns
    ]
    .sum(
        axis=1
    )
)


zero_feature_mask = (
    genome_feature_counts
    == 0
)


zero_feature_genomes = (
    df_x_final_19.loc[
        zero_feature_mask,
        "Genome ID"
    ]
    .tolist()
)


# =========================================================
# 8. Matrix density / sparsity
# =========================================================

total_cells = (
    n_genomes
    * n_features
)


present_cells = int(
    df_x_final_19[
        feature_columns
    ]
    .to_numpy()
    .sum()
)


density = (
    present_cells
    / total_cells
)


sparsity = (
    1
    - density
)


# =========================================================
# 9. Check known technical artifact
# =========================================================

emrd_present = any(
    str(col)
    .strip()
    .lower()
    == "emrd"
    for col in feature_columns
)


# =========================================================
# 10. Ensure identifiers / phenotype are not predictive X
# =========================================================

forbidden_predictor_names = {
    "Genome ID",
    "Genome Name",
    "Taxon ID",
    "Resistant Phenotype"
}


forbidden_features_found = (
    forbidden_predictor_names
    .intersection(
        set(
            feature_columns
        )
    )
)


# =========================================================
# 11. Build audit record
# =========================================================

audit_records = [
    {
        "Metric":
            "Study genomes",

        "Value":
            n_genomes
    },
    {
        "Metric":
            "Unique Genome IDs",

        "Value":
            n_unique_genomes
    },
    {
        "Metric":
            "Final known-AMR features",

        "Value":
            n_features
    },
    {
        "Metric":
            "Missing study genomes",

        "Value":
            len(
                missing_genomes
            )
    },
    {
        "Metric":
            "Unexpected genomes",

        "Value":
            len(
                unexpected_genomes
            )
    },
    {
        "Metric":
            "Missing matrix values",

        "Value":
            missing_values
    },
    {
        "Metric":
            "Constant features",

        "Value":
            len(
                constant_features
            )
    },
    {
        "Metric":
            "Zero-feature genomes",

        "Value":
            len(
                zero_feature_genomes
            )
    },
    {
        "Metric":
            "Matrix density",

        "Value":
            density
    },
    {
        "Metric":
            "Matrix sparsity",

        "Value":
            sparsity
    },
    {
        "Metric":
            "emrD present",

        "Value":
            emrd_present
    },
    {
        "Metric":
            "Forbidden predictor columns",

        "Value":
            len(
                forbidden_features_found
            )
    }
]


df_final_audit = pd.DataFrame(
    audit_records
)


# =========================================================
# 12. Hard integrity assertions
# =========================================================

assert n_genomes == 4233

assert n_unique_genomes == 4233

assert n_features == 668

assert len(
    missing_genomes
) == 0

assert len(
    unexpected_genomes
) == 0

assert missing_values == 0

assert binary_valid

assert not emrd_present

assert len(
    forbidden_features_found
) == 0


# =========================================================
# 13. Save audit
# =========================================================

df_final_audit.to_csv(
    FINAL_AUDIT_PATH,
    index=False
)


# =========================================================
# 14. Summary
# =========================================================

print("=" * 70)
print("FINAL KNOWN-AMR MATRIX INTEGRITY AUDIT")
print("=" * 70)

print(
    f"Study genomes          : "
    f"{n_genomes:,}"
)

print(
    f"Unique Genome IDs      : "
    f"{n_unique_genomes:,}"
)

print(
    f"Known-AMR features     : "
    f"{n_features:,}"
)

print(
    f"Missing matrix values  : "
    f"{missing_values:,}"
)

print(
    f"Constant features      : "
    f"{len(constant_features):,}"
)

print(
    f"Zero-feature genomes   : "
    f"{len(zero_feature_genomes):,}"
)

print(
    f"Matrix density         : "
    f"{density:.4%}"
)

print(
    f"Matrix sparsity        : "
    f"{sparsity:.4%}"
)

print(
    f"emrD present           : "
    f"{emrd_present}"
)

print(
    f"Forbidden predictors   : "
    f"{len(forbidden_features_found)}"
)

print()


if constant_features:

    print(
        "Constant features:"
    )

    print(
        constant_features
    )

    print()


if zero_feature_genomes:

    print(
        "Genome IDs with zero retained known-AMR features:"
    )

    print(
        zero_feature_genomes
    )

    print()


print(
    "Audit saved to:"
)

print(
    FINAL_AUDIT_PATH
)

print()

print(
    "PASS: The finalized known-AMR feature matrix "
    "passed the File 04 integrity audit."
)

FINAL KNOWN-AMR MATRIX INTEGRITY AUDIT
Study genomes          : 4,233
Unique Genome IDs      : 4,233
Known-AMR features     : 668
Missing matrix values  : 0
Constant features      : 0
Zero-feature genomes   : 3
Matrix density         : 2.2430%
Matrix sparsity        : 97.7570%
emrD present           : False
Forbidden predictors   : 0

Genome IDs with zero retained known-AMR features:
['573.15368', '573.15452', '573.15459']

Audit saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/known_amr_final_integrity_audit.csv

PASS: The finalized known-AMR feature matrix passed the File 04 integrity audit.


# Cell 20 — Finalize File 04 Known-AMR Feature Engineering

In [9]:
import os
import pandas as pd
import numpy as np


# =========================================================
# 1. Persistent final-output paths
# =========================================================

X_KNOWN_AMR_FINAL_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "X_known_amr_final.csv"
)

FINAL_FEATURE_METADATA_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_feature_metadata_final.csv"
)

FINAL_AUDIT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_final_integrity_audit.csv"
)

FINAL_DECISION_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "known_amr_final_harmonization_decision.csv"
)

FILE04_SUMMARY_PATH = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "file04_known_amr_feature_engineering_summary.csv"
)


# =========================================================
# 2. Verify required outputs exist
# =========================================================

required_outputs = [
    X_KNOWN_AMR_FINAL_PATH,
    FINAL_FEATURE_METADATA_PATH,
    FINAL_AUDIT_PATH,
    FINAL_DECISION_PATH
]


missing_outputs = [
    path
    for path in required_outputs
    if not os.path.exists(path)
]


assert not missing_outputs, (
    "Missing final File 04 outputs:\n"
    + "\n".join(missing_outputs)
)


# =========================================================
# 3. Reload finalized feature matrix
# =========================================================

df_x_final_20 = pd.read_csv(
    X_KNOWN_AMR_FINAL_PATH,
    dtype={
        "Genome ID": str
    }
)


df_x_final_20[
    "Genome ID"
] = (
    df_x_final_20[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


feature_columns = [
    col
    for col in df_x_final_20.columns
    if col != "Genome ID"
]


# =========================================================
# 4. Final integrity confirmation
# =========================================================

assert len(df_x_final_20) == 4233

assert (
    df_x_final_20[
        "Genome ID"
    ]
    .nunique()
    == 4233
)

assert len(feature_columns) == 668

assert (
    df_x_final_20[
        feature_columns
    ]
    .isna()
    .sum()
    .sum()
    == 0
)


unique_values = set(
    np.unique(
        df_x_final_20[
            feature_columns
        ].values
    )
)


assert unique_values.issubset(
    {
        0,
        1
    }
)


assert "emrD" not in feature_columns


# =========================================================
# 5. Final descriptive values
# =========================================================

genome_feature_counts = (
    df_x_final_20[
        feature_columns
    ]
    .sum(axis=1)
)


zero_feature_genomes = int(
    (
        genome_feature_counts
        == 0
    )
    .sum()
)


matrix_density = float(
    df_x_final_20[
        feature_columns
    ]
    .to_numpy()
    .sum()
    /
    (
        len(df_x_final_20)
        * len(feature_columns)
    )
)


# =========================================================
# 6. Create File 04 decision record
# =========================================================

df_file04_summary = pd.DataFrame(
    [
        {
            "File":
                "04_known_amr_feature_engineering",

            "Status":
                "FINALIZED",

            "Study Genomes":
                4233,

            "NCBI AMR Source Genomes":
                3851,

            "Local AMRFinder Genomes":
                382,

            "AMRFinderPlus Version":
                "4.2.7",

            "Local Reference DB":
                "2026-08-07.1",

            "Local Screening Mode":
                "NUCLEOTIDE",

            "Initial Features":
                669,

            "Technical Features Removed":
                1,

            "Removed Technical Feature":
                "emrD",

            "Final Features":
                668,

            "Feature Encoding":
                "Binary presence/absence",

            "Zero-feature Genomes":
                zero_feature_genomes,

            "Matrix Density":
                matrix_density,

            "Phenotype Used for Feature Definition":
                False,

            "Supervised Feature Selection Performed":
                False,

            "Final Matrix":
                X_KNOWN_AMR_FINAL_PATH
        }
    ]
)


df_file04_summary.to_csv(
    FILE04_SUMMARY_PATH,
    index=False
)


# =========================================================
# 7. Final report
# =========================================================

print("=" * 70)
print("FILE 04 — FINAL STATUS")
print("=" * 70)

print(
    f"Study genomes             : "
    f"{len(df_x_final_20):,}"
)

print(
    f"NCBI AMR genomes          : "
    f"{3_851:,}"
)

print(
    f"Local AMRFinder genomes   : "
    f"{382:,}"
)

print()

print(
    f"Initial known-AMR features: "
    f"{669:,}"
)

print(
    f"Technical artifact removed: "
    f"emrD"
)

print(
    f"Final known-AMR features  : "
    f"{len(feature_columns):,}"
)

print(
    f"Final matrix              : "
    f"{len(df_x_final_20):,} x "
    f"{len(feature_columns):,}"
)

print(
    f"Zero-feature genomes      : "
    f"{zero_feature_genomes:,}"
)

print(
    f"Matrix density            : "
    f"{matrix_density:.4%}"
)

print()

print(
    "Phenotype used for feature definition : NO"
)

print(
    "Supervised feature selection          : NOT YET"
)

print()

print(
    "Final feature matrix:"
)

print(
    X_KNOWN_AMR_FINAL_PATH
)

print()

print(
    "File 04 summary:"
)

print(
    FILE04_SUMMARY_PATH
)

print()

print(
    "PASS: File 04 known-AMR feature engineering "
    "is finalized and locked."
)

FILE 04 — FINAL STATUS
Study genomes             : 4,233
NCBI AMR genomes          : 3,851
Local AMRFinder genomes   : 382

Initial known-AMR features: 669
Technical artifact removed: emrD
Final known-AMR features  : 668
Final matrix              : 4,233 x 668
Zero-feature genomes      : 3
Matrix density            : 2.2430%

Phenotype used for feature definition : NO
Supervised feature selection          : NOT YET

Final feature matrix:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/X_known_amr_final.csv

File 04 summary:
/content/drive/MyDrive/AMR-Genome-ML/checkpoints/file04_known_amr_feature_engineering_summary.csv

PASS: File 04 known-AMR feature engineering is finalized and locked.
